# Revisi eksperimen PAPER024914 — versi Google Colab + Google Drive

Notebook ini mengulang protokol yang sama dengan eksperimen asli (split 80/20 seed 42, MinMax hanya pada train, test tidak diubah, CTGAN hanya pada Web Attack train, undersample Benign, class weight, CNN-LSTM), lalu menambahkan uji yang diminta reviewer. **Semua keluaran ditulis ke Google Drive**, supaya hasil tidak hilang jika sesi Colab terputus. Angka makalah hanya boleh diambil dari mode `full` atau `reduced` pada data asli, bukan dari mode `smoke`.

## Struktur folder di Google Drive

Unggah tiga CSV harian ke folder berikut (nama folder dan nama file harus sama, kecuali Anda mengubah `DRIVE_ROOT` di sel berikutnya):

```
MyDrive/MECS_Revisi/
  Data/
    02-14-2018.csv
    02-16-2018.csv
    02-22-2018.csv
  results/          ← dibuat otomatis (jangan campur hasil laptop ke sini)
```

`DRIVE_ROOT` default: `/content/drive/MyDrive/MECS_Revisi`. Ubah **satu variabel itu saja** jika Anda memakai path lain.

## Langkah menjalankan

1. Di Colab, pilih **Runtime → Change runtime type → GPU**. Untuk paket Google AI Pro (200 CU), **A100 40 GB** paling nyaman; jika tidak tersedia, pakai **L4**, lalu **T4**. Hindari A100 80 GB kecuali benar-benar diperlukan (tarif CU lebih mahal).
2. **Jangan biarkan tab tertutup atau laptop sleep.** Paket ini tidak punya background execution. Satu sesi biasanya sekitar 12 jam.
3. Jalankan sel dari atas ke bawah. Sel pertama akan meminta izin merakit Google Drive.
4. Jika diminta restart runtime setelah `pip install`, restart, lalu jalankan lagi dari sel *mount Drive* (sel `pip` boleh dijalankan ulang; itu aman).
5. Kalau sesi terputus: buka notebook ini lagi, pilih GPU yang sama jika memungkinkan, jalankan semua sel dari atas. Pelatihan yang sudah `status=ok` di `results/all_runs.csv` akan **dilewati otomatis**. Cache sintetis CTGAN, file `proba_*.npz`, dan log juga dibaca dari Drive, jadi analisis ambang/filter/mutu bisa dijalankan di sesi berikutnya tanpa melatih ulang.
6. Jangan mencampur angka dari laptop (mode `reduced`, 15 epoch, CTGAN 50 epoch) dengan angka Colab di naskah. Semua tabel paper sebaiknya berasal dari **satu mesin Colab**.

## Mode

- `full` (default di notebook ini): 5 seed `[42, 123, 456, 789, 1024]`, CTGAN 200 epoch, pelatihan CNN-LSTM 50 epoch (early stopping patience 5), variasi sintetis 1000/2500/5000/10000. Ada **28 kali pelatihan**. `RUN_CTGAN_VARIANT` default `False`.
- `reduced`: 2 seed, 15 epoch, CTGAN 50 epoch — hanya jika CU atau waktu tersisa sangat sedikit; sebutkan di surat reviewer jika dipakai.
- `smoke`: data palsu, **jangan dilaporkan ke jurnal**.

## Perkiraan waktu (kasar)

Satu pelatihan di A100 sering belasan menit, di L4 bisa lebih lama. 28 run + analisis biasanya muat di 200 CU, tetapi bisa memakan lebih dari satu sesi 12 jam — itulah gunanya *resume*.


In [15]:
# ============================================================
# Mount Google Drive dan atur folder kerja
# Ubah HANYA variabel DRIVE_ROOT jika path Drive Anda berbeda.
# ============================================================
from pathlib import Path
import os
import shutil
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/MECS_Revisi")

EXPECTED_CSVS = ["02-14-2018.csv", "02-16-2018.csv", "02-22-2018.csv"]
COPY_CSV_TO_LOCAL = True          # salin ke disk lokal Colab agar pembacaan lebih cepat
LOCAL_DATA_DIR = Path("/content/cic_data")

# Mount (aman dijalankan ulang setelah reconnect)
try:
    from google.colab import drive as _colab_drive
    IN_COLAB = True
except ImportError:
    _colab_drive = None
    IN_COLAB = False
    print("Bukan runtime Google Colab. Drive tidak di-mount. "
          "Untuk uji lokal, isi DRIVE_ROOT ke folder yang berisi Data/.")

if IN_COLAB:
    mount_point = Path("/content/drive")
    if (mount_point / "MyDrive").exists():
        print("Google Drive sudah ter-mount.")
    else:
        _colab_drive.mount("/content/drive")

DATA_DIR_DRIVE = DRIVE_ROOT / "Data"
RESULTS_DIR = DRIVE_ROOT / "results"
RUN_LOG_DIR = RESULTS_DIR / "run_logs"
FIG_PAPER_DIR = RESULTS_DIR / "figures_paper"
for _p in (RESULTS_DIR, RUN_LOG_DIR, FIG_PAPER_DIR):
    _p.mkdir(parents=True, exist_ok=True)


def _norm_name(name: str) -> str:
    return name.strip().lower()


def inspect_data_folder(folder: Path):
    """Periksa tiga CSV resmi; kembalikan peta nama resmi -> path, plus daftar file yang ditemukan."""
    folder = Path(folder)
    found_files = sorted(folder.glob("*")) if folder.exists() else []
    found_names = [p.name for p in found_files if p.is_file()]
    csv_names = [n for n in found_names if n.lower().endswith(".csv")]
    by_lower = {p.name.lower(): p for p in found_files if p.is_file()}
    mapping = {}
    missing = []
    for official in EXPECTED_CSVS:
        hit = by_lower.get(official.lower())
        if hit is None:
            missing.append(official)
        else:
            mapping[official] = hit
    return mapping, missing, csv_names, found_names


if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Folder DRIVE_ROOT belum ada: {}\n"
        "Buat folder itu di Google Drive, lalu unggah subfolder Data/ berisi tiga CSV harian:\n"
        "  {}".format(DRIVE_ROOT, ", ".join(EXPECTED_CSVS))
    )

if not DATA_DIR_DRIVE.exists():
    raise FileNotFoundError(
        "Folder data tidak ditemukan: {}\n"
        "Buat folder 'Data' di dalam {} dan unggah ketiga CSV berikut:\n  {}".format(
            DATA_DIR_DRIVE, DRIVE_ROOT, ", ".join(EXPECTED_CSVS)
        )
    )

mapping, missing, csv_names, found_names = inspect_data_folder(DATA_DIR_DRIVE)
if missing:
    daftar_ditemukan = ", ".join(found_names) if found_names else "(folder kosong atau tidak ada berkas)"
    daftar_csv = ", ".join(csv_names) if csv_names else "(tidak ada file .csv)"
    raise FileNotFoundError(
        "Tiga CSV CSE-CIC-IDS2018 yang diperlukan belum lengkap di {folder}.\n"
        "Yang wajib ada:\n  {wajib}\n"
        "Yang belum ketemu:\n  {kurang}\n"
        "File yang ditemukan di folder itu:\n  {semua}\n"
        "File CSV yang ditemukan:\n  {csv}\n"
        "Pastikan nama file tepat (02-14-2018.csv, 02-16-2018.csv, 02-22-2018.csv) "
        "dan berada langsung di dalam folder Data, bukan di subfolder lain."
        .format(
            folder=DATA_DIR_DRIVE,
            wajib=", ".join(EXPECTED_CSVS),
            kurang=", ".join(missing),
            semua=daftar_ditemukan,
            csv=daftar_csv,
        )
    )

extra_csv = [n for n in csv_names if _norm_name(n) not in {_norm_name(x) for x in EXPECTED_CSVS}]
if extra_csv:
    print("PERINGATAN: ada CSV lain di Data/ selain tiga hari resmi:", extra_csv)
    print("Fungsi pemuat akan membaca SEMUA CSV di folder. Pindahkan file ekstra agar protokol tetap 3 hari.")

# Salin ke disk lokal (opsional)
DATA_DIR = DATA_DIR_DRIVE
if COPY_CSV_TO_LOCAL:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    copied = []
    for official, src in mapping.items():
        dst = LOCAL_DATA_DIR / official
        need = (not dst.exists()) or (dst.stat().st_size != src.stat().st_size)
        if need:
            print(f"Menyalin {src.name} -> {dst} ...")
            shutil.copy2(src, dst)
        copied.append(dst)
        print(f"  {official}: {dst} ({dst.stat().st_size:,} byte)")
    DATA_DIR = LOCAL_DATA_DIR
    print("DATA_DIR memakai salinan lokal:", DATA_DIR)
else:
    print("DATA_DIR memakai folder Drive langsung:", DATA_DIR)

print("DRIVE_ROOT  =", DRIVE_ROOT)
print("RESULTS_DIR =", RESULTS_DIR)
print("Tiga CSV siap:")
for official, src in mapping.items():
    used = DATA_DIR / official
    print(f"  {official}: {used} ({used.stat().st_size:,} byte)")


Google Drive sudah ter-mount.
  02-14-2018.csv: /content/cic_data/02-14-2018.csv (358,223,333 byte)
  02-16-2018.csv: /content/cic_data/02-16-2018.csv (333,723,605 byte)
  02-22-2018.csv: /content/cic_data/02-22-2018.csv (382,636,202 byte)
DATA_DIR memakai salinan lokal: /content/cic_data
DRIVE_ROOT  = /content/drive/MyDrive/Colab Notebooks/MECS_Revisi
RESULTS_DIR = /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results
Tiga CSV siap:
  02-14-2018.csv: /content/cic_data/02-14-2018.csv (358,223,333 byte)
  02-16-2018.csv: /content/cic_data/02-16-2018.csv (333,723,605 byte)
  02-22-2018.csv: /content/cic_data/02-22-2018.csv (382,636,202 byte)


In [16]:
# ============================================================
# Instal pustaka tambahan. TensorFlow sudah ada di Colab — jangan di-pin ulang.
# ============================================================
import sys
import subprocess

pkgs = ["ctgan", "imbalanced-learn", "lightgbm", "psutil"]
print("Memasang:", ", ".join(pkgs))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Instalasi selesai. Jika import di sel berikutnya gagal, pilih Runtime → Restart session,")
print("lalu jalankan lagi mulai sel mount Drive (sel ini aman dijalankan ulang).")


Memasang: ctgan, imbalanced-learn, lightgbm, psutil
Instalasi selesai. Jika import di sel berikutnya gagal, pilih Runtime → Restart session,
lalu jalankan lagi mulai sel mount Drive (sel ini aman dijalankan ulang).


In [17]:
# ============================================================
# KONFIGURASI — ubah di sini sebelum menjalankan
# Path DATA_DIR / RESULTS_DIR sudah diatur di sel mount Drive.
# Jangan memakai os.environ.get untuk RUN_MODE (menghindari jebakan variabel lingkungan).
# ============================================================
from pathlib import Path

# "smoke" | "reduced" | "full"
RUN_MODE = "full"

# Protokol asli
N_SYNTH_DEFAULT = 5000
SPLIT_SEED = 42
TEST_SIZE = 0.20
CTGAN_EPOCHS_DEFAULT = 200
CTGAN_BATCH_SIZE = 50
CTGAN_GENERATOR_DIM = (256, 256)
CTGAN_DISCRIMINATOR_DIM = (256, 256)
CTGAN_PAC = 10
DL_EPOCHS_DEFAULT = 50
DL_BATCH_SIZE = 512          # tetap 512 agar sebanding dengan run asli
DL_PATIENCE = 5
SEEDS_FULL = [42, 123, 456, 789, 1024]
SYNTH_COUNTS_FULL = [1000, 2500, 5000, 10000]
THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

# Mode reduced: potong komputasi, tetap protokol evaluasi yang sama pada test utuh
SEEDS_REDUCED = [42, 123]
SYNTH_COUNTS_REDUCED = [5000]
DL_EPOCHS_REDUCED = 15
DL_PATIENCE_REDUCED = 3
CTGAN_EPOCHS_REDUCED = 50

# Mode smoke (data palsu, jangan dilaporkan ke jurnal)
SEEDS_SMOKE = [42]
SYNTH_COUNTS_SMOKE = [40]
DL_EPOCHS_SMOKE = 1
DL_BATCH_SMOKE = 32
DL_PATIENCE_SMOKE = 1
CTGAN_EPOCHS_SMOKE = 2
CTGAN_BATCH_SMOKE = 10
N_SYNTH_SMOKE = 40

# Flag eksperimen (boleh dimatikan manual)
RUN_BASELINES = True          # no-aug, SMOTE, ADASYN, Borderline-SMOTE
RUN_ABLATION_ARCH = True      # CNN-only, LSTM-only
RUN_ABLATION_WEIGHTS = True   # tanpa class weight
RUN_ABLATION_SYNTH_N = True   # 1000/2500/5000/10000 (tergantung mode)
RUN_MULTI_SEED = True
RUN_THRESHOLD = True
RUN_SECONDARY_FILTER = True
RUN_CTGAN_QUALITY = True
RUN_CTGAN_VARIANT = False     # default mati; nyalakan nanti jika CU masih longgar
RUN_ROBUST_SCALER = True      # hitung overflow MinMax vs RobustScaler (tanpa latih ulang, default)
USE_LIGHTGBM_IF_AVAILABLE = True
USE_MIXED_PRECISION = False   # biarkan False supaya hasil sebanding dengan run float32 asli

# Jika True, mode smoke tidak memanggil CTGAN.fit (pakai jitter). Mode full/reduced tetap CTGAN.
SMOKE_SKIP_CTGAN_FIT = True

# Pastikan folder hasil ada (sel mount sudah membuatnya di Colab)
RESULTS_DIR = Path(RESULTS_DIR)
RUN_LOG_DIR = Path(RESULTS_DIR) / "run_logs"
FIG_PAPER_DIR = Path(RESULTS_DIR) / "figures_paper"
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)
FIG_PAPER_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_MODE    = {RUN_MODE!r}")
print(f"DATA_DIR    = {Path(DATA_DIR).resolve()}")
print(f"RESULTS_DIR = {Path(RESULTS_DIR).resolve()}")
print(f"USE_MIXED_PRECISION = {USE_MIXED_PRECISION}")
print(f"DL_BATCH_SIZE = {DL_BATCH_SIZE}")


RUN_MODE    = 'full'
DATA_DIR    = /content/cic_data
RESULTS_DIR = /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results
USE_MIXED_PRECISION = False
DL_BATCH_SIZE = 512


In [18]:
# ============================================================
# Import — selaras dengan notebook asli, plus pustaka revisi
# ============================================================
import os
import gc
import json
import time
import random
import platform
import traceback
import warnings
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set(style="whitegrid")

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from scipy import stats as scipy_stats

try:
    import torch
except ImportError:
    torch = None

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Flatten, GlobalAveragePooling1D,
    )
    from tensorflow.keras.utils import to_categorical
    from tensorflow.keras.callbacks import EarlyStopping
    TF_AVAILABLE = True
except ImportError as exc:
    tf = None
    TF_AVAILABLE = False
    print("TensorFlow belum terpasang:", exc)

try:
    from ctgan import CTGAN
    CTGAN_AVAILABLE = True
except ImportError as exc:
    CTGAN = None
    CTGAN_AVAILABLE = False
    print("CTGAN belum terpasang:", exc)

try:
    from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
    IMBLEARN_AVAILABLE = True
except ImportError as exc:
    IMBLEARN_AVAILABLE = False
    print("imbalanced-learn belum terpasang:", exc)

LGBM_AVAILABLE = False
if USE_LIGHTGBM_IF_AVAILABLE:
    try:
        from lightgbm import LGBMClassifier
        LGBM_AVAILABLE = True
    except ImportError:
        LGBM_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string())
        else:
            print(obj)

print("Import selesai.")

print("TF:", TF_AVAILABLE, "| CTGAN:", CTGAN_AVAILABLE, "| imblearn:", IMBLEARN_AVAILABLE, "| LightGBM:", LGBM_AVAILABLE)

try:
    from IPython import get_ipython as _get_ipython
    _ip = _get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except Exception:
    pass


Import selesai.
TF: True | CTGAN: True | imblearn: True | LightGBM: True


In [19]:
# ============================================================
# Log hardware / lingkungan Colab, pertumbuhan memori GPU, mixed precision
# ============================================================
import subprocess

def _run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True, timeout=30)
    except Exception as exc:
        return f"(gagal menjalankan {cmd!r}: {exc})"


def configure_tf_gpu():
    """Memory growth ON; mixed precision default OFF (flag USE_MIXED_PRECISION)."""
    if not TF_AVAILABLE:
        return {"tf_gpu_configured": False}
    info = {"tf_gpu_configured": True, "memory_growth": True, "mixed_precision": bool(USE_MIXED_PRECISION)}
    try:
        gpus = tf.config.list_physical_devices("GPU")
        info["physical_gpus"] = [g.name for g in gpus]
        for g in gpus:
            try:
                tf.config.experimental.set_memory_growth(g, True)
            except Exception as exc:
                print("set_memory_growth gagal (mungkin sudah diinisialisasi):", exc)
        if USE_MIXED_PRECISION:
            tf.keras.mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision AKTIF (mixed_float16). Hasil bisa berbeda dari run asli.")
        else:
            try:
                tf.keras.mixed_precision.set_global_policy("float32")
            except Exception:
                pass
            print("Mixed precision NONAKTIF (float32), agar sebanding dengan eksperimen asli.")
    except Exception as exc:
        info["tf_gpu_error"] = str(exc)
    return info


def log_environment(save_path=RESULTS_DIR / "environment.json"):
    info = {
        "timestamp_iso": datetime.now().isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_count_logical": os.cpu_count(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "matplotlib": plt.matplotlib.__version__,
        "tensorflow": getattr(tf, "__version__", None) if TF_AVAILABLE else None,
        "torch": getattr(torch, "__version__", None) if torch is not None else None,
        "ctgan_available": CTGAN_AVAILABLE,
        "imblearn_available": IMBLEARN_AVAILABLE,
        "lightgbm_available": LGBM_AVAILABLE,
        "run_mode": RUN_MODE,
        "drive_root": str(DRIVE_ROOT),
        "data_dir": str(Path(DATA_DIR).resolve()),
        "results_dir": str(Path(RESULTS_DIR).resolve()),
        "in_colab": bool(globals().get("IN_COLAB", False)),
        "use_mixed_precision": bool(USE_MIXED_PRECISION),
        "dl_batch_size": int(DL_BATCH_SIZE),
    }
    try:
        import psutil
        vm = psutil.virtual_memory()
        info["ram_total_gb"] = round(vm.total / (1024 ** 3), 2)
        info["ram_available_gb"] = round(vm.available / (1024 ** 3), 2)
        try:
            info["cpu_freq_mhz"] = getattr(psutil.cpu_freq(), "current", None)
        except Exception:
            pass
    except ImportError:
        info["ram_total_gb"] = None
        info["note_psutil"] = "psutil tidak terpasang; RAM tidak tercatat"

    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        for line in cpuinfo.read_text(errors="replace").splitlines():
            if "model name" in line:
                info["cpu_model"] = line.split(":", 1)[1].strip()
                break

    meminfo = Path("/proc/meminfo")
    if meminfo.exists():
        for line in meminfo.read_text(errors="replace").splitlines():
            if line.startswith("MemTotal:"):
                kb = int(line.split()[1])
                info["ram_total_gb_proc"] = round(kb / (1024 ** 2), 2)
                break

    print("----- nvidia-smi -----")
    smi = _run_cmd(["nvidia-smi"])
    print(smi)
    info["nvidia_smi"] = smi[-4000:] if isinstance(smi, str) else str(smi)

    info["tf_gpus"] = []
    if TF_AVAILABLE:
        try:
            gpus = tf.config.list_physical_devices("GPU")
            info["tf_gpus"] = [g.name for g in gpus]
            info["tf_built_with_cuda"] = bool(tf.test.is_built_with_cuda())
            for i, g in enumerate(gpus):
                try:
                    det = tf.config.experimental.get_device_details(g)
                    info.setdefault("tf_gpu_details", []).append({"index": i, **{str(k): str(v) for k, v in det.items()}})
                    print(f"TF GPU {i}: {g.name} {det}")
                except Exception as exc:
                    print("Detail GPU TF tidak terbaca:", exc)
        except Exception as exc:
            info["tf_gpu_error"] = str(exc)
    info["torch_cuda"] = bool(torch.cuda.is_available()) if torch is not None else False
    if torch is not None and torch.cuda.is_available():
        try:
            info["torch_gpu_name"] = torch.cuda.get_device_name(0)
            print("PyTorch GPU:", info["torch_gpu_name"])
        except Exception:
            pass

    info.update(configure_tf_gpu())

    save_path = Path(save_path)
    save_path.write_text(json.dumps(info, indent=2))
    # ringkas tanpa dump nvidia-smi penuh
    slim = {k: v for k, v in info.items() if k != "nvidia_smi"}
    print(json.dumps(slim, indent=2))
    print(f"Tersimpan: {save_path}")
    return info

ENV_INFO = log_environment()


----- nvidia-smi -----
Sat Nov 14 10:15:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             42W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------------

## Praproses — disalin dari notebook asli (Tahap 2–9, 12)

Perubahan hanya pada path data, fungsi yang dapat dipanggil ulang, dan mode `smoke`. Split tetap 80/20, `random_state=42`, `stratify=y` untuk protokol utama (seed lain hanya untuk ulang pelatihan / CTGAN / undersample, lihat komentar di sel pelatihan).

**Kebijakan perbandingan (penting):**

- `main_ctgan`: seperti asli — CTGAN \(N\) pada Web Attack train, undersample Benign = jumlah semua serangan (asli+sintetis), class weight, CNN-LSTM.
- `undersample_only` (ablasi tanpa CTGAN): tanpa sintetis, undersample Benign = jumlah serangan asli, class weight, CNN-LSTM.
- `weights_only` (baseline tanpa augmentasi): train penuh, tanpa sintetis, tanpa undersample, hanya class weight, CNN-LSTM.
- `smote` / `adasyn` / `borderline_smote`: oversample Web Attack train saja sampai \(290+N\) (atau setara di smoke), lalu undersample Benign dengan aturan yang sama seperti `main_ctgan`.


In [20]:
# 68 nama fitur setelah buang kolom konstan di eksperimen asli (urutan cicflowmeter, huruf kecil)
FEATURE_COLS_68 = [
    "dst port", "protocol", "flow duration", "tot fwd pkts", "tot bwd pkts",
    "totlen fwd pkts", "totlen bwd pkts", "fwd pkt len max", "fwd pkt len min",
    "fwd pkt len mean", "fwd pkt len std", "bwd pkt len max", "bwd pkt len min",
    "bwd pkt len mean", "bwd pkt len std", "flow byts/s", "flow pkts/s",
    "flow iat mean", "flow iat std", "flow iat max", "flow iat min",
    "fwd iat tot", "fwd iat mean", "fwd iat std", "fwd iat max", "fwd iat min",
    "bwd iat tot", "bwd iat mean", "bwd iat std", "bwd iat max", "bwd iat min",
    "fwd psh flags", "fwd header len", "bwd header len", "fwd pkts/s", "bwd pkts/s",
    "pkt len min", "pkt len max", "pkt len mean", "pkt len std", "pkt len var",
    "fin flag cnt", "syn flag cnt", "rst flag cnt", "psh flag cnt", "ack flag cnt",
    "urg flag cnt", "ece flag cnt", "down/up ratio", "pkt size avg",
    "fwd seg size avg", "bwd seg size avg", "subflow fwd pkts", "subflow fwd byts",
    "subflow bwd pkts", "subflow bwd byts", "init fwd win byts", "init bwd win byts",
    "fwd act data pkts", "fwd seg size min", "active mean", "active std",
    "active max", "active min", "idle mean", "idle std", "idle max", "idle min",
]
assert len(FEATURE_COLS_68) == 68, len(FEATURE_COLS_68)

KOLOM_METADATA = ["flow id", "src ip", "dst ip", "src port", "timestamp"]
WEB_LABELS = ["Brute Force -Web", "Brute Force -XSS", "SQL Injection"]
CLASS_ORDER = [
    "Benign",
    "DoS attacks-Hulk",
    "DoS attacks-SlowHTTPTest",
    "FTP-BruteForce",
    "SSH-Bruteforce",
    "Web Attack",
]
CONSTANT_COLS_EXPECTED = [
    "bwd psh flags", "fwd urg flags", "bwd urg flags", "cwe flag count",
    "fwd byts/b avg", "fwd pkts/b avg", "fwd blk rate avg",
    "bwd byts/b avg", "bwd pkts/b avg", "bwd blk rate avg",
]


def set_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    if TF_AVAILABLE:
        tf.random.set_seed(seed)


def load_cic_dataset(base_path: Path) -> pd.DataFrame:
    """Tahap 2 asli: muat semua CSV, buang metadata, downcast, buang inf/NaN/header anomali."""
    import glob
    base_path = Path(base_path)
    all_files = sorted(glob.glob(str(base_path / "*.csv")))
    print(f"Ditemukan {len(all_files)} file CSV di {base_path}")
    if not all_files:
        raise FileNotFoundError(f"Tidak ada CSV di {base_path}")

    df_list = []
    for file_path in all_files:
        file_name = os.path.basename(file_path)
        print(f"Memproses {file_name}...")
        temp_df = pd.read_csv(file_path, low_memory=False)
        temp_df.columns = temp_df.columns.str.strip().str.lower()
        temp_df = temp_df.drop(
            columns=[c for c in KOLOM_METADATA if c in temp_df.columns],
            errors="ignore",
        )
        for col in temp_df.select_dtypes(include=["float64"]).columns:
            temp_df[col] = temp_df[col].astype("float32")
        for col in temp_df.select_dtypes(include=["int64"]).columns:
            temp_df[col] = pd.to_numeric(temp_df[col], downcast="integer")
        df_list.append(temp_df)
        gc.collect()

    print("Menggabungkan dataset...")
    df = pd.concat(df_list, ignore_index=True)
    del df_list
    gc.collect()

    if "label" in df.columns:
        df = df[df["label"] != "Label"]
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    if "label" in df.columns:
        df.rename(columns={"label": "Label"}, inplace=True)
    print(f"Total baris bersih: {df.shape[0]:,} | kolom: {df.shape[1]}")
    return df


def clean_and_map_multiclass(df: pd.DataFrame) -> pd.DataFrame:
    """Tahap 4–5 asli: numerik, buang kolom konstan, gabung Web Attack."""
    df = df.copy()
    df["Label"] = df["Label"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    fitur = [c for c in df.columns if c != "Label"]
    df[fitur] = df[fitur].apply(pd.to_numeric, errors="coerce")
    n_nan = int(df[fitur].isna().sum().sum())
    print(f"NaN setelah konversi numerik: {n_nan:,}")
    if n_nan > 0:
        df.dropna(subset=fitur, inplace=True)
    nunique = df[fitur].nunique(dropna=False)
    kolom_konstan = nunique[nunique <= 1].index.tolist()
    print("Kolom konstan dibuang:", kolom_konstan)
    if kolom_konstan:
        df.drop(columns=kolom_konstan, inplace=True)
    df["Label"] = df["Label"].replace({k: "Web Attack" for k in WEB_LABELS})
    print("Distribusi setelah gabung Web Attack:")
    print(df["Label"].value_counts())
    return df


def make_smoke_dataframe(n_features=68, seed=42) -> pd.DataFrame:
    """Data palsu dengan 68 kolom dan 6 kelas — hanya untuk mengetes alur."""
    rng = np.random.default_rng(seed)
    sizes = {
        "Benign": 480,
        "DoS attacks-Hulk": 240,
        "FTP-BruteForce": 120,
        "SSH-Bruteforce": 120,
        "DoS attacks-SlowHTTPTest": 120,
        "Web Attack": 48,
    }
    rows = []
    for i, (lab, n) in enumerate(sizes.items()):
        mean = rng.normal(i * 0.4, 1.0, size=n_features)
        x = rng.normal(0, 1, size=(n, n_features)) + mean
        # sedikit nilai ekstrem agar overflow MinMax bisa muncul di 'test'
        if lab == "Benign":
            x[:3] *= 8
        part = pd.DataFrame(x, columns=FEATURE_COLS_68)
        part["Label"] = lab
        rows.append(part)
    df = pd.concat(rows, ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    print(f"[SMOKE] Data sintetis palsu: {df.shape[0]} baris. JANGAN dipakai di makalah.")
    return df


In [21]:
# ============================================================
# Split, scaler, overflow, balancing — protokol asli
# ============================================================
def split_and_scale(X, y, scaler_kind="minmax", split_seed=SPLIT_SEED, test_size=TEST_SIZE):
    print(f"Train-test split {1-test_size:.0%}/{test_size:.0%}, stratify=y, random_state={split_seed}")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=split_seed, stratify=y,
    )
    print(f"Train: {X_train.shape[0]:,} x {X_train.shape[1]} | Test: {X_test.shape[0]:,} x {X_test.shape[1]}")
    print("Distribusi TRAIN:\n", y_train.value_counts())
    print("Distribusi TEST (jangan diubah):\n", y_test.value_counts())

    if scaler_kind == "minmax":
        scaler = MinMaxScaler()
    elif scaler_kind == "robust":
        scaler = RobustScaler()
    else:
        raise ValueError(scaler_kind)

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    feature_cols = list(X_train.columns)
    X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols).reset_index(drop=True)
    X_test_df = pd.DataFrame(X_test_scaled, columns=feature_cols).reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)
    y_test = pd.Series(y_test).reset_index(drop=True)
    return scaler, X_train_df, X_test_df, y_train, y_test


def count_minmax_overflow(X_test_df: pd.DataFrame, tag="minmax"):
    """Hitung nilai test di luar [0, 1] setelah scaler yang di-fit di train."""
    arr = X_test_df.to_numpy()
    mask = (arr < 0) | (arr > 1)
    per_col = pd.Series(mask.sum(axis=0), index=X_test_df.columns, name="n_overflow")
    rec = {
        "scaler": tag,
        "n_values": int(arr.size),
        "n_values_lt0": int((arr < 0).sum()),
        "n_values_gt1": int((arr > 1).sum()),
        "n_values_outside_0_1": int(mask.sum()),
        "n_rows": int(arr.shape[0]),
        "n_rows_with_overflow": int(mask.any(axis=1).sum()),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "top_features": per_col.sort_values(ascending=False).head(15).to_dict(),
    }
    per_col.sort_values(ascending=False).to_csv(RESULTS_DIR / f"overflow_per_feature_{tag}.csv")
    pd.DataFrame([rec]).drop(columns=["top_features"]).to_csv(
        RESULTS_DIR / f"overflow_summary_{tag}.csv", index=False
    )
    print(json.dumps({k: rec[k] for k in rec if k != "top_features"}, indent=2))
    print("Fitur overflow terbanyak:", rec["top_features"])
    return rec


def undersample_benign(train_df: pd.DataFrame, random_state=42) -> pd.DataFrame:
    """Seperti Tahap 12: Benign di-sample sebanyak jumlah baris non-Benign."""
    df_benign = train_df[train_df["Label"] == "Benign"]
    df_attack = train_df[train_df["Label"] != "Benign"]
    n = len(df_attack)
    if n == 0:
        raise ValueError("Tidak ada baris serangan untuk undersample Benign.")
    if len(df_benign) < n:
        print(f"Peringatan: Benign ({len(df_benign)}) < serangan ({n}); pakai semua Benign.")
        df_b = df_benign
    else:
        df_b = df_benign.sample(n=n, random_state=random_state)
    out = pd.concat([df_b, df_attack], ignore_index=True)
    return out.sample(frac=1, random_state=random_state).reset_index(drop=True)


def attach_synth_web(X_train_df, y_train, X_web_synth) -> pd.DataFrame:
    train_combined = X_train_df.copy()
    train_combined["Label"] = y_train.values
    synth_df = X_web_synth.copy()
    synth_df["Label"] = "Web Attack"
    return pd.concat([train_combined, synth_df], ignore_index=True)


In [22]:
# ============================================================
# CTGAN, oversampling klasik, mutu sintetis
# ============================================================
def jitter_synth(X_real: pd.DataFrame, n_synth: int, seed=42) -> pd.DataFrame:
    """Cadangan smoke saja: salinan + noise. Bukan CTGAN."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(X_real), size=n_synth)
    arr = X_real.to_numpy()[idx].astype(np.float64)
    arr = arr + rng.normal(0, 0.02, size=arr.shape)
    return pd.DataFrame(arr, columns=X_real.columns)


def fit_ctgan_and_sample(X_web_gan: pd.DataFrame, n_synth: int, seed=42,
                         epochs=None, batch_size=None, generator_dim=None,
                         discriminator_dim=None, pac=None, learning_rate=None,
                         tag="default"):
    """Latih CTGAN hanya pada Web Attack train, lalu sample n_synth baris."""
    set_seeds(seed)
    epochs = CTGAN_EPOCHS if epochs is None else epochs
    batch_size = CTGAN_BATCH_SIZE if batch_size is None else batch_size
    # batch_size harus < n_baris dan kelipatan pac
    pac = CTGAN_PAC if pac is None else pac
    if batch_size >= len(X_web_gan):
        batch_size = max(pac, (len(X_web_gan) // pac) * pac)
        if batch_size < pac:
            batch_size = pac
        print(f"batch_size CTGAN disesuaikan ke {batch_size}")

    kwargs = dict(
        epochs=epochs,
        batch_size=batch_size,
        generator_dim=generator_dim or CTGAN_GENERATOR_DIM,
        discriminator_dim=discriminator_dim or CTGAN_DISCRIMINATOR_DIM,
        pac=pac,
        verbose=True,
    )
    # Beberapa versi CTGAN memakai 'discriminator_lr' / 'generator_lr'
    if learning_rate is not None:
        kwargs["discriminator_lr"] = learning_rate
        kwargs["generator_lr"] = learning_rate

    t0 = time.perf_counter()
    model = CTGAN(**kwargs)
    model.fit(X_web_gan)
    fit_s = time.perf_counter() - t0
    set_seeds(seed)
    X_synth = model.sample(n_synth)
    X_synth.columns = X_web_gan.columns
    X_synth = X_synth.astype(np.float64)
    print(f"CTGAN[{tag}] selesai fit {fit_s:.1f}s; sample {X_synth.shape}")

    # simpan kurva loss jika ada
    try:
        loss_df = model.loss_values.copy()
        loss_df.to_csv(RESULTS_DIR / f"ctgan_loss_{tag}_seed{seed}.csv", index=False)
        plt.figure(figsize=(10, 5))
        plt.plot(loss_df["Epoch"], loss_df["Generator Loss"], label="Generator Loss")
        plt.plot(loss_df["Epoch"], loss_df["Discriminator Loss"], label="Discriminator Loss")
        plt.title(f"CTGAN loss — {tag}, seed={seed}")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True, linestyle="--", alpha=0.7)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / f"ctgan_loss_{tag}_seed{seed}.png", dpi=300, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        print("Loss CTGAN tidak tersimpan:", exc)
        loss_df = None
    return model, X_synth, {"fit_seconds": fit_s, "epochs": epochs, "batch_size": batch_size, "tag": tag}


def oversample_web_imblearn(X_train_df, y_train, method: str, n_synth: int, seed=42):
    """Oversample kelas Web Attack saja hingga n_real + n_synth."""
    if not IMBLEARN_AVAILABLE:
        raise ImportError("imbalanced-learn diperlukan untuk SMOTE/ADASYN/Borderline-SMOTE")
    y = pd.Series(y_train).reset_index(drop=True)
    X = X_train_df.reset_index(drop=True)
    n_real = int((y == "Web Attack").sum())
    target = n_real + int(n_synth)
    k = min(5, max(1, n_real - 1))
    sampling = {lab: int((y == lab).sum()) for lab in y.unique()}
    sampling["Web Attack"] = target
    common = dict(random_state=seed, sampling_strategy=sampling)
    if method == "smote":
        sampler = SMOTE(k_neighbors=k, **common)
    elif method == "adasyn":
        sampler = ADASYN(n_neighbors=k, **common)
    elif method == "borderline_smote":
        sampler = BorderlineSMOTE(k_neighbors=k, **common)
    else:
        raise ValueError(method)
    X_res, y_res = sampler.fit_resample(X, y)
    X_res = pd.DataFrame(X_res, columns=X.columns)
    y_res = pd.Series(y_res, name="Label")
    print(f"{method}: Web Attack {n_real} -> {(y_res=='Web Attack').sum()} (target {target})")
    return X_res, y_res


def evaluate_ctgan_quality(X_real: pd.DataFrame, X_synth: pd.DataFrame, tag="default", seed=42):
    """Selisih korelasi, KS per fitur, diskriminator real-vs-synth, ringkasan TSTR kasar."""
    cols = list(X_real.columns)
    real = X_real[cols].astype(float)
    synth = X_synth[cols].astype(float)
    corr_r = real.corr().fillna(0)
    corr_s = synth.corr().fillna(0)
    diff = (corr_r - corr_s).abs()
    corr_r.to_csv(RESULTS_DIR / f"corr_real_{tag}.csv")
    corr_s.to_csv(RESULTS_DIR / f"corr_synth_{tag}.csv")
    diff.to_csv(RESULTS_DIR / f"corr_absdiff_{tag}.csv")

    plt.figure(figsize=(8, 6))
    # heatmap 12 fitur pertama agar terbaca; matriks penuh tetap di CSV
    k = min(12, len(cols))
    sns.heatmap(diff.iloc[:k, :k], cmap="mako", vmin=0, vmax=1)
    plt.title(f"Absolute correlation difference (first {k} features) — {tag}")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"corr_absdiff_{tag}.png", dpi=300, bbox_inches="tight")
    plt.close()

    ks_rows = []
    for c in cols:
        try:
            stat, p = scipy_stats.ks_2samp(real[c].to_numpy(), synth[c].to_numpy())
        except Exception:
            stat, p = np.nan, np.nan
        ks_rows.append({"feature": c, "ks_statistic": stat, "p_value": p,
                        "real_mean": real[c].mean(), "synth_mean": synth[c].mean()})
    ks_df = pd.DataFrame(ks_rows).sort_values("ks_statistic", ascending=False)
    ks_df.to_csv(RESULTS_DIR / f"ks_per_feature_{tag}.csv", index=False)

    # Diskriminator: 1=real, 0=synth (sample seimbang)
    rng = np.random.default_rng(seed)
    n = min(len(real), len(synth), 4000)
    ri = rng.choice(len(real), n, replace=False)
    si = rng.choice(len(synth), n, replace=False)
    Xd = pd.concat([real.iloc[ri], synth.iloc[si]], ignore_index=True)
    yd = np.array([1] * n + [0] * n)
    Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=seed, stratify=yd)
    clf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    clf.fit(Xd_tr, yd_tr)
    proba = clf.predict_proba(Xd_te)[:, 1]
    disc_auc = float(roc_auc_score(yd_te, proba))
    disc_acc = float(accuracy_score(yd_te, (proba >= 0.5).astype(int)))

    summary = pd.DataFrame([{
        "tag": tag,
        "n_real": len(real),
        "n_synth": len(synth),
        "corr_absdiff_mean": float(diff.to_numpy()[np.triu_indices_from(diff, k=1)].mean()) if len(cols) > 1 else 0.0,
        "corr_absdiff_frobenius": float(np.linalg.norm(diff.to_numpy())),
        "ks_mean": float(ks_df["ks_statistic"].mean()),
        "ks_median": float(ks_df["ks_statistic"].median()),
        "ks_max": float(ks_df["ks_statistic"].max()),
        "ks_max_feature": ks_df.iloc[0]["feature"] if len(ks_df) else None,
        "discriminator_auc": disc_auc,
        "discriminator_acc": disc_acc,
        "note": "AUC diskriminator mendekati 0.5 berarti real/synth sulit dibedakan; mendekati 1 berarti mudah dibedakan (mutu rendah).",
    }])
    summary.to_csv(RESULTS_DIR / f"ctgan_quality_summary_{tag}.csv", index=False)
    print(summary.T)
    return summary, ks_df


In [23]:
# ============================================================
# Model CNN-LSTM (arsitektur asli) + varian ablasi
# ============================================================
def resolve_mode_params():
    if RUN_MODE == "full":
        return dict(
            seeds=list(SEEDS_FULL),
            synth_counts=list(SYNTH_COUNTS_FULL),
            n_synth=N_SYNTH_DEFAULT,
            dl_epochs=DL_EPOCHS_DEFAULT,
            dl_batch=DL_BATCH_SIZE,
            dl_patience=DL_PATIENCE,
            ctgan_epochs=CTGAN_EPOCHS_DEFAULT,
            ctgan_batch=CTGAN_BATCH_SIZE,
            use_real_data=True,
        )
    if RUN_MODE == "reduced":
        return dict(
            seeds=list(SEEDS_REDUCED),
            synth_counts=list(SYNTH_COUNTS_REDUCED),
            n_synth=N_SYNTH_DEFAULT,
            dl_epochs=DL_EPOCHS_REDUCED,
            dl_batch=DL_BATCH_SIZE,
            dl_patience=DL_PATIENCE_REDUCED,
            ctgan_epochs=CTGAN_EPOCHS_REDUCED,
            ctgan_batch=CTGAN_BATCH_SIZE,
            use_real_data=True,
        )
    # smoke
    return dict(
        seeds=list(SEEDS_SMOKE),
        synth_counts=list(SYNTH_COUNTS_SMOKE),
        n_synth=N_SYNTH_SMOKE,
        dl_epochs=DL_EPOCHS_SMOKE,
        dl_batch=DL_BATCH_SMOKE,
        dl_patience=DL_PATIENCE_SMOKE,
        ctgan_epochs=CTGAN_EPOCHS_SMOKE,
        ctgan_batch=CTGAN_BATCH_SMOKE,
        use_real_data=False,
    )


MODE = resolve_mode_params()
# alias yang dipakai fungsi CTGAN di sel sebelumnya
CTGAN_EPOCHS = MODE["ctgan_epochs"]
print("Parameter mode:", MODE)


def build_model(n_timesteps: int, n_classes: int, arch="cnn_lstm"):
    """arch: cnn_lstm (asli) | cnn | lstm"""
    if not TF_AVAILABLE:
        raise RuntimeError("TensorFlow diperlukan untuk membangun model.")
    model = Sequential()
    if arch == "cnn_lstm":
        # Identik dengan sel 14 notebook asli
        model.add(Conv1D(64, 3, activation="relu", input_shape=(n_timesteps, 1)))
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.3))
        model.add(LSTM(128, return_sequences=True))
        model.add(Dropout(0.3))
        model.add(LSTM(64))
        model.add(Dense(64, activation="relu"))
        model.add(Dropout(0.3))
        model.add(Dense(n_classes, activation="softmax"))
    elif arch == "cnn":
        model.add(Conv1D(64, 3, activation="relu", input_shape=(n_timesteps, 1)))
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.3))
        model.add(Conv1D(128, 3, activation="relu"))
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.3))
        model.add(GlobalAveragePooling1D())
        model.add(Dense(64, activation="relu"))
        model.add(Dropout(0.3))
        model.add(Dense(n_classes, activation="softmax"))
    elif arch == "lstm":
        model.add(LSTM(128, return_sequences=True, input_shape=(n_timesteps, 1)))
        model.add(Dropout(0.3))
        model.add(LSTM(64))
        model.add(Dense(64, activation="relu"))
        model.add(Dropout(0.3))
        model.add(Dense(n_classes, activation="softmax"))
    else:
        raise ValueError(arch)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def metrics_from_preds(y_true_enc, y_pred_class, class_names, y_proba=None, prefix=""):
    report = classification_report(
        y_true_enc, y_pred_class, target_names=class_names, digits=4, zero_division=0, output_dict=True,
    )
    acc = accuracy_score(y_true_enc, y_pred_class)
    f1_macro = f1_score(y_true_enc, y_pred_class, average="macro", zero_division=0)
    f1_weighted = f1_score(y_true_enc, y_pred_class, average="weighted", zero_division=0)
    rec = {
        f"{prefix}accuracy": acc,
        f"{prefix}f1_macro": f1_macro,
        f"{prefix}f1_weighted": f1_weighted,
    }
    for cname in class_names:
        rec[f"{prefix}{cname}__precision"] = report[cname]["precision"]
        rec[f"{prefix}{cname}__recall"] = report[cname]["recall"]
        rec[f"{prefix}{cname}__f1"] = report[cname]["f1-score"]
        rec[f"{prefix}{cname}__support"] = report[cname]["support"]
    return rec, report


def save_confusion(y_true_enc, y_pred_class, class_names, path, title):
    cm = confusion_matrix(y_true_enc, y_pred_class)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(path.with_suffix(".csv"))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.close()
    return cm


def measure_inference_latency(model, X_test_dl, batch_size, n_warmup=3):
    """Wall-clock prediksi test + perkiraan latensi per sampel (setelah warmup kecil)."""
    # warmup
    n_w = min(len(X_test_dl), max(batch_size, 32))
    for _ in range(n_warmup):
        _ = model.predict(X_test_dl[:n_w], batch_size=batch_size, verbose=0)
    t0 = time.perf_counter()
    proba = model.predict(X_test_dl, batch_size=batch_size, verbose=0)
    elapsed = time.perf_counter() - t0
    per_sample_ms = (elapsed / len(X_test_dl)) * 1000.0
    return proba, elapsed, per_sample_ms


def train_eval_dl(
    X_train_df, y_train, X_test_df, y_test,
    arch="cnn_lstm", use_class_weight=True, seed=42,
    run_name="run", epochs=None, batch_size=None, patience=None,
):
    if not TF_AVAILABLE:
        raise RuntimeError("TensorFlow diperlukan.")
    set_seeds(seed)
    epochs = MODE["dl_epochs"] if epochs is None else epochs
    batch_size = MODE["dl_batch"] if batch_size is None else batch_size
    patience = MODE["dl_patience"] if patience is None else patience

    le = LabelEncoder()
    le.fit(CLASS_ORDER)  # urutan tetap, sama dengan run asli
    # jaga-jaga jika smoke kehilangan kelas
    present = list(pd.Series(y_train).unique()) + list(pd.Series(y_test).unique())
    if any(c not in le.classes_ for c in present):
        le = LabelEncoder()
        le.fit(sorted(set(present)))

    y_train_enc = le.transform(y_train)
    y_test_enc = le.transform(y_test)
    n_classes = len(le.classes_)
    y_train_cat = to_categorical(y_train_enc, num_classes=n_classes)
    X_train_dl = np.asarray(X_train_df, dtype=np.float32).reshape((len(X_train_df), X_train_df.shape[1], 1))
    X_test_dl = np.asarray(X_test_df, dtype=np.float32).reshape((len(X_test_df), X_test_df.shape[1], 1))

    class_weight = None
    if use_class_weight:
        weights = compute_class_weight("balanced", classes=np.unique(y_train_enc), y=y_train_enc)
        class_weight = {i: w for i, w in zip(np.unique(y_train_enc), weights)}
        print("class_weight:", {le.classes_[i]: round(float(w), 3) for i, w in class_weight.items()})

    model = build_model(X_train_dl.shape[1], n_classes, arch=arch)
    early_stop = EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
    t0 = time.perf_counter()
    history = model.fit(
        X_train_dl, y_train_cat,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=1,
    )
    train_s = time.perf_counter() - t0
    n_epochs_ran = len(history.history.get("loss", []))
    print(f"[{run_name}] pelatihan {train_s/60:.2f} menit, {n_epochs_ran} epoch (max {epochs})")

    proba, infer_s, infer_ms = measure_inference_latency(model, X_test_dl, batch_size)
    y_pred = proba.argmax(axis=1)
    rec, report = metrics_from_preds(y_test_enc, y_pred, list(le.classes_))
    rec.update({
        "run_name": run_name,
        "arch": arch,
        "seed": seed,
        "use_class_weight": use_class_weight,
        "n_train": int(len(X_train_df)),
        "n_test": int(len(X_test_df)),
        "train_seconds": train_s,
        "n_epochs_ran": n_epochs_ran,
        "inference_seconds": infer_s,
        "inference_ms_per_sample": infer_ms,
        "run_mode": RUN_MODE,
    })
    print(classification_report(y_test_enc, y_pred, target_names=le.classes_, digits=4, zero_division=0))
    print(f"Macro F1={rec['f1_macro']:.4f} | Web F1={rec.get('Web Attack__f1', float('nan')):.4f} | "
          f"Web P={rec.get('Web Attack__precision', float('nan')):.4f} | infer={infer_ms:.4f} ms/sampel")

    hist_df = pd.DataFrame(history.history)
    hist_df.to_csv(RESULTS_DIR / f"history_{run_name}.csv", index=False)
    plt.figure(figsize=(10, 5))
    if "accuracy" in hist_df:
        plt.plot(hist_df["accuracy"], label="Train Accuracy")
        plt.plot(hist_df["val_accuracy"], label="Val Accuracy", linestyle="--")
    plt.plot(hist_df["loss"], label="Train Loss")
    plt.plot(hist_df["val_loss"], label="Val Loss", linestyle="--")
    plt.title(run_name)
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.7)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"history_{run_name}.png", dpi=300, bbox_inches="tight")
    plt.close()

    save_confusion(
        y_test_enc, y_pred, list(le.classes_),
        RESULTS_DIR / f"cm_{run_name}",
        f"Confusion matrix — {run_name} (test untouched)",
    )
    # simpan proba untuk analisis ambang (bisa besar; di smoke/reduced OK)
    proba_path = RESULTS_DIR / f"proba_{run_name}.npz"
    np.savez_compressed(
        proba_path,
        proba=proba.astype(np.float32),
        y_true=y_test_enc,
        classes=np.array(le.classes_),
    )
    rec["proba_path"] = str(proba_path)
    # bersihkan graf TF
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()
    return rec, model, le, proba, y_test_enc


Parameter mode: {'seeds': [42, 123, 456, 789, 1024], 'synth_counts': [1000, 2500, 5000, 10000], 'n_synth': 5000, 'dl_epochs': 50, 'dl_batch': 512, 'dl_patience': 5, 'ctgan_epochs': 200, 'ctgan_batch': 50, 'use_real_data': True}


In [24]:
# ============================================================
# Muat data, split seed 42, MinMax, hitung overflow, opsi RobustScaler
# ============================================================
set_seeds(SPLIT_SEED)
IS_SMOKE = False

if RUN_MODE == "smoke" or not MODE["use_real_data"]:
    IS_SMOKE = True
    df_model = make_smoke_dataframe(seed=SPLIT_SEED)
    X = df_model[FEATURE_COLS_68]
    y = df_model["Label"]
else:
    if not Path(DATA_DIR).exists() or not any(Path(DATA_DIR).glob("*.csv")):
        raise FileNotFoundError(
            f"DATA_DIR={DATA_DIR} tidak berisi CSV. "
            "Mode full/reduced tidak boleh beralih ke data palsu. "
            "Periksa folder Data di Google Drive dan jalankan ulang sel mount."
        )
    df_raw = load_cic_dataset(DATA_DIR)
    df_model = clean_and_map_multiclass(df_raw)
    missing = [c for c in FEATURE_COLS_68 if c not in df_model.columns]
    extra = [c for c in df_model.columns if c not in FEATURE_COLS_68 and c != "Label"]
    print("Fitur yang diharapkan tetapi hilang:", missing)
    print("Fitur ekstra (tetap dipakai jika ada):", extra)
    feature_cols = [c for c in df_model.columns if c != "Label"]
    X = df_model[feature_cols]
    y = df_model["Label"]

scaler, X_train_mm, X_test_mm, y_train, y_test = split_and_scale(
    X, y, scaler_kind="minmax", split_seed=SPLIT_SEED,
)
print(f"Rentang train min/max: {X_train_mm.min().min():.4f} / {X_train_mm.max().max():.4f}")
print(f"Rentang test  min/max: {X_test_mm.min().min():.4f} / {X_test_mm.max().max():.4f}")
overflow_mm = count_minmax_overflow(X_test_mm, tag="minmax")

if RUN_ROBUST_SCALER:
    scaler_rob, X_train_rob, X_test_rob, y_train_r, y_test_r = split_and_scale(
        X, y, scaler_kind="robust", split_seed=SPLIT_SEED,
    )
    # Overflow [0,1] pada RobustScaler tidak bermakna sama, tetapi tetap dihitung agar transparan
    overflow_rob = count_minmax_overflow(X_test_rob, tag="robust")
    print("RobustScaler test min/max:", float(X_test_rob.min().min()), float(X_test_rob.max().max()))
else:
    X_train_rob = X_test_rob = None

# Bahan CTGAN: Web Attack pada train (skala MinMax), seperti Tahap 8
X_web_gan = X_train_mm[y_train.values == "Web Attack"].copy()
print(f"Bahan CTGAN Web Attack: {X_web_gan.shape[0]} baris x {X_web_gan.shape[1]} fitur")
X_web_gan.to_csv(RESULTS_DIR / "X_web_gan.csv", index=False)
y_train.to_csv(RESULTS_DIR / "y_train.csv", index=False)
y_test.to_csv(RESULTS_DIR / "y_test.csv", index=False)


Ditemukan 3 file CSV di /content/cic_data
Memproses 02-14-2018.csv...
Memproses 02-16-2018.csv...
Memproses 02-22-2018.csv...
Menggabungkan dataset...
Total baris bersih: 3,136,290 | kolom: 79
NaN setelah konversi numerik: 0
Kolom konstan dibuang: ['bwd psh flags', 'fwd urg flags', 'bwd urg flags', 'cwe flag count', 'fwd byts/b avg', 'fwd pkts/b avg', 'fwd blk rate avg', 'bwd byts/b avg', 'bwd pkts/b avg', 'bwd blk rate avg']
Distribusi setelah gabung Web Attack:
Label
Benign                      2153183
DoS attacks-Hulk             461912
FTP-BruteForce               193354
SSH-Bruteforce               187589
DoS attacks-SlowHTTPTest     139890
Web Attack                      362
Name: count, dtype: int64
Fitur yang diharapkan tetapi hilang: []
Fitur ekstra (tetap dipakai jika ada): []
Train-test split 80%/20%, stratify=y, random_state=42
Train: 2,509,032 x 68 | Test: 627,258 x 68
Distribusi TRAIN:
 Label
Benign                      1722546
DoS attacks-Hulk             369530
FTP-Brut

In [25]:
# ============================================================
# Bangun data train untuk tiap skenario (test TIDAK diubah)
# ============================================================
def build_train_for_scenario(scenario, n_synth, seed, X_synth_ctgan=None):
    """Kembalikan X_train_bal, y_train_bal, meta."""
    meta = {"scenario": scenario, "n_synth": n_synth, "seed": seed}
    if scenario == "weights_only":
        # tanpa sintetis, tanpa undersample
        return X_train_mm, y_train, {**meta, "n_train": len(X_train_mm)}

    if scenario == "undersample_only":
        train_combined = X_train_mm.copy()
        train_combined["Label"] = y_train.values
        bal = undersample_benign(train_combined, random_state=seed)
        return bal.drop(columns=["Label"]), bal["Label"], {**meta, "n_train": len(bal)}

    if scenario == "main_ctgan":
        if X_synth_ctgan is None:
            raise ValueError("X_synth_ctgan wajib untuk main_ctgan")
        # pastikan jumlah baris sesuai permintaan (potong atau peringatkan)
        Xs = X_synth_ctgan.iloc[:n_synth].copy() if len(X_synth_ctgan) >= n_synth else X_synth_ctgan
        if len(Xs) != n_synth:
            print(f"Peringatan: sintetis tersedia {len(Xs)}, diminta {n_synth}")
        combined = attach_synth_web(X_train_mm, y_train, Xs)
        bal = undersample_benign(combined, random_state=seed)
        print("Distribusi train (main_ctgan):\n", bal["Label"].value_counts())
        return bal.drop(columns=["Label"]), bal["Label"], {**meta, "n_train": len(bal), "n_synth_used": len(Xs)}

    if scenario in {"smote", "adasyn", "borderline_smote"}:
        X_res, y_res = oversample_web_imblearn(X_train_mm, y_train, scenario, n_synth, seed=seed)
        combined = X_res.copy()
        combined["Label"] = y_res.values
        bal = undersample_benign(combined, random_state=seed)
        print(f"Distribusi train ({scenario}):\n", bal["Label"].value_counts())
        return bal.drop(columns=["Label"]), bal["Label"], {**meta, "n_train": len(bal)}

    raise ValueError(scenario)


def get_or_fit_ctgan_synth(n_synth, seed, tag="default", **ctgan_kwargs):
    cache = RESULTS_DIR / f"X_web_synth_{tag}_seed{seed}_n{n_synth}.csv"
    if cache.exists():
        print(f"Memakai sintetis ter-cache: {cache}")
        return pd.read_csv(cache), {"cached": True, "tag": tag}

    use_jitter = (IS_SMOKE or RUN_MODE == "smoke") and SMOKE_SKIP_CTGAN_FIT
    if use_jitter or not CTGAN_AVAILABLE:
        if not use_jitter:
            print("CTGAN tidak tersedia — TIDAK boleh dipakai sebagai hasil makalah.")
        else:
            print("[SMOKE] Menggunakan jitter, bukan CTGAN.fit.")
        Xs = jitter_synth(X_web_gan, n_synth, seed=seed)
        Xs.to_csv(cache, index=False)
        return Xs, {"cached": False, "method": "jitter", "tag": tag}

    _, Xs, meta = fit_ctgan_and_sample(X_web_gan, n_synth, seed=seed, tag=tag, **ctgan_kwargs)
    Xs.to_csv(cache, index=False)
    return Xs, meta


In [26]:
# ============================================================
# Runner eksperimen — baseline, ablasi, multi-seed, dengan RESUME
# Semua evaluasi hanya pada X_test_mm / y_test yang sama (split seed 42).
# Seed memengaruhi inisialisasi model, undersample, CTGAN/SMOTE, BUKAN pecahan test.
# Jika sesi putus: jalankan ulang sel ini; run dengan status=ok dilewati.
# ============================================================
import json as _json

ALL_RUNS_CSV = RESULTS_DIR / "all_runs.csv"


def _coerce_records(df):
    if df is None or len(df) == 0:
        return []
    return df.to_dict(orient="records")


def load_runs_csv():
    if ALL_RUNS_CSV.exists():
        df = pd.read_csv(ALL_RUNS_CSV)
        print(f"Memuat {ALL_RUNS_CSV} ({len(df)} baris).")
        if "status" in df.columns:
            n_ok = int(df["status"].eq("ok").sum())
            n_err = int(df["status"].eq("error").sum())
            print(f"  status ok={n_ok}, error={n_err}")
        return df
    print("Belum ada all_runs.csv — mulai dari awal.")
    return pd.DataFrame()


def persist_runs(rows):
    df = pd.DataFrame(rows)
    if len(df) and "run_name" in df.columns:
        if "status" in df.columns:
            df["_ok"] = df["status"].eq("ok").astype(int)
        else:
            df["_ok"] = 0
        df = df.sort_values("_ok").drop_duplicates("run_name", keep="last")
        df = df.drop(columns=["_ok"])
    df.to_csv(ALL_RUNS_CSV, index=False)
    return df


def find_ok_row(run_name, rows=None):
    src = rows
    if src is None:
        df = load_runs_csv() if ALL_RUNS_CSV.exists() else pd.DataFrame()
        src = _coerce_records(df)
    for rec in src:
        if rec.get("run_name") == run_name and str(rec.get("status", "")).lower() == "ok":
            return rec
    return None


def write_run_log(run_name, payload):
    RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)
    path = RUN_LOG_DIR / f"{run_name}.json"
    blob = dict(payload)
    blob["logged_iso"] = datetime.now().isoformat(timespec="seconds")
    # nilai yang tidak JSON-serializable dibuang
    clean = {}
    for k, v in blob.items():
        try:
            _json.dumps(v)
            clean[k] = v
        except TypeError:
            clean[k] = str(v)
    path.write_text(_json.dumps(clean, indent=2))
    return path


def planned_jobs():
    """Daftar pekerjaan pelatihan, urutan sama dengan notebook asli."""
    jobs = []
    seeds = list(MODE["seeds"])
    n_default = MODE["n_synth"]
    # 1. model utama seed pertama
    jobs.append(dict(scenario="main_ctgan", seed=seeds[0], arch="cnn_lstm",
                     use_class_weight=True, n_synth=n_default, tag_extra=""))
    if RUN_BASELINES:
        for scen in ["weights_only", "undersample_only", "smote", "adasyn", "borderline_smote"]:
            jobs.append(dict(scenario=scen, seed=seeds[0], arch="cnn_lstm",
                             use_class_weight=True, n_synth=n_default, tag_extra=""))
    if RUN_ABLATION_ARCH:
        jobs.append(dict(scenario="main_ctgan", seed=seeds[0], arch="cnn",
                         use_class_weight=True, n_synth=n_default, tag_extra="_ablation"))
        jobs.append(dict(scenario="main_ctgan", seed=seeds[0], arch="lstm",
                         use_class_weight=True, n_synth=n_default, tag_extra="_ablation"))
    if RUN_ABLATION_WEIGHTS:
        jobs.append(dict(scenario="main_ctgan", seed=seeds[0], arch="cnn_lstm",
                         use_class_weight=False, n_synth=n_default, tag_extra="_noweight"))
    if RUN_ABLATION_SYNTH_N:
        for n in MODE["synth_counts"]:
            if n == n_default:
                continue
            jobs.append(dict(scenario="main_ctgan", seed=seeds[0], arch="cnn_lstm",
                             use_class_weight=True, n_synth=n, tag_extra="_nablate"))
    if RUN_MULTI_SEED:
        key_scen = ["main_ctgan", "weights_only", "undersample_only", "smote"]
        if not RUN_BASELINES:
            key_scen = ["main_ctgan"]
        for seed in seeds:
            if seed == seeds[0]:
                continue
            for scen in key_scen:
                jobs.append(dict(scenario=scen, seed=seed, arch="cnn_lstm",
                                 use_class_weight=True, n_synth=n_default, tag_extra=""))
    return jobs


def make_run_name(scenario, seed, arch="cnn_lstm", use_class_weight=True, n_synth=None, tag_extra=""):
    n_synth = MODE["n_synth"] if n_synth is None else n_synth
    return f"{scenario}_{arch}_w{int(use_class_weight)}_n{n_synth}_seed{seed}{tag_extra}"


def maybe_eval_quality(X_synth, seed, n_synth, tag_extra):
    if not RUN_CTGAN_QUALITY:
        return
    if seed != MODE["seeds"][0] or n_synth != MODE["n_synth"] or tag_extra != "":
        return
    tag = f"n{n_synth}_seed{seed}"
    summary_path = RESULTS_DIR / f"ctgan_quality_summary_{tag}.csv"
    if summary_path.exists():
        print(f"Mutu CTGAN sudah ada ({summary_path.name}); dilewati.")
        return
    try:
        evaluate_ctgan_quality(X_web_gan, X_synth, tag=tag, seed=seed)
    except Exception:
        traceback.print_exc()


all_rows = _coerce_records(load_runs_csv())


def run_one(scenario, seed, arch="cnn_lstm", use_class_weight=True, n_synth=None, tag_extra="",
            progress=None):
    n_synth = MODE["n_synth"] if n_synth is None else n_synth
    run_name = make_run_name(scenario, seed, arch, use_class_weight, n_synth, tag_extra)
    prefix = f"[{progress}] " if progress else ""
    print("\n" + "=" * 70)
    print(f"{prefix}RUN {run_name}")
    print("=" * 70)

    existing = find_ok_row(run_name, all_rows)
    if existing is not None:
        print(f"{prefix}SKIP — sudah status=ok di all_runs.csv")
        write_run_log(run_name, {**existing, "skipped": True})
        # pastikan baris ini ada di all_rows
        if not any(r.get("run_name") == run_name and str(r.get("status", "")).lower() == "ok" for r in all_rows):
            all_rows.append(existing)
        persist_runs(all_rows)
        return existing, None, None, None, None

    X_synth = None
    synth_meta = {}
    try:
        if scenario == "main_ctgan":
            X_synth, synth_meta = get_or_fit_ctgan_synth(n_synth, seed, tag=f"n{n_synth}{tag_extra}")
            maybe_eval_quality(X_synth, seed, n_synth, tag_extra)
        Xb, yb, meta = build_train_for_scenario(scenario, n_synth, seed, X_synth_ctgan=X_synth)
        rec, model, le, proba, y_test_enc = train_eval_dl(
            Xb, yb, X_test_mm, y_test,
            arch=arch, use_class_weight=use_class_weight, seed=seed, run_name=run_name,
        )
        rec.update(meta)
        rec.update({f"synth_{k}": v for k, v in synth_meta.items() if not isinstance(v, (dict, list))})
        rec["status"] = "ok"
        all_rows.append(rec)
        persist_runs(all_rows)
        write_run_log(run_name, rec)
        return rec, model, le, proba, y_test_enc
    except Exception as exc:
        traceback.print_exc()
        rec = {"run_name": run_name, "scenario": scenario, "seed": seed, "arch": arch,
               "use_class_weight": use_class_weight, "n_synth": n_synth,
               "status": "error", "error": str(exc)}
        all_rows.append(rec)
        persist_runs(all_rows)
        write_run_log(run_name, rec)
        return rec, None, None, None, None


JOBS = planned_jobs()
N_JOBS = len(JOBS)
already_ok = sum(1 for j in JOBS if find_ok_row(make_run_name(**j), all_rows) is not None)
print(f"Total run terencana: {N_JOBS} (mode={RUN_MODE!r}). Sudah selesai: {already_ok}. Sisa: {N_JOBS - already_ok}.")
for i, job in enumerate(JOBS, start=1):
    print(f"\n>>> run {i} of {N_JOBS}")
    run_one(progress=f"run {i} of {N_JOBS}", **job)

persist_runs(all_rows)
print("Selesai batch pelatihan. Jumlah baris:", len(pd.DataFrame(all_rows).drop_duplicates("run_name") if all_rows else []))
_df_show = persist_runs(all_rows)
cols_show = [c for c in [
    "run_name", "scenario", "arch", "seed", "n_synth", "status",
    "accuracy", "f1_macro", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1",
    "train_seconds", "inference_ms_per_sample",
] if c in _df_show.columns]
display(_df_show[cols_show] if len(_df_show) else _df_show)
print("File hasil ada di Drive:", ALL_RUNS_CSV)
print("Jika sesi akan habis, aman memutus sekarang. Sambungkan lagi dan jalankan sel dari atas; sisa run akan dilanjutkan.")


Memuat /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results/all_runs.csv (28 baris).
  status ok=28, error=0
Total run terencana: 28 (mode='full'). Sudah selesai: 28. Sisa: 0.

>>> run 1 of 28

[run 1 of 28] RUN main_ctgan_cnn_lstm_w1_n5000_seed42
[run 1 of 28] SKIP — sudah status=ok di all_runs.csv

>>> run 2 of 28

[run 2 of 28] RUN weights_only_cnn_lstm_w1_n5000_seed42
[run 2 of 28] SKIP — sudah status=ok di all_runs.csv

>>> run 3 of 28

[run 3 of 28] RUN undersample_only_cnn_lstm_w1_n5000_seed42
[run 3 of 28] SKIP — sudah status=ok di all_runs.csv

>>> run 4 of 28

[run 4 of 28] RUN smote_cnn_lstm_w1_n5000_seed42
[run 4 of 28] SKIP — sudah status=ok di all_runs.csv

>>> run 5 of 28

[run 5 of 28] RUN adasyn_cnn_lstm_w1_n5000_seed42
[run 5 of 28] SKIP — sudah status=ok di all_runs.csv

>>> run 6 of 28

[run 6 of 28] RUN borderline_smote_cnn_lstm_w1_n5000_seed42
[run 6 of 28] SKIP — sudah status=ok di all_runs.csv

>>> run 7 of 28

[run 7 of 28] RUN main_ctgan_cnn_w1_n5000_see

,run_name,scenario,arch,seed,n_synth,status,accuracy,f1_macro,Web Attack__precision,Web Attack__recall,Web Attack__f1,train_seconds,inference_ms_per_sample
0,main_ctgan_cnn_lstm_w1_n5000_seed42,main_ctgan,cnn_lstm,42,5000,ok,0.9625,0.8406,0.6207,1.0000,0.7660,1246.6712,0.0072
1,weights_only_cnn_lstm_w1_n5000_seed42,weights_only,cnn_lstm,42,5000,ok,0.9537,0.7118,0.0275,1.0000,0.0536,517.6188,0.0070
2,undersample_only_cnn_lstm_w1_n5000_seed42,undersample_only,cnn_lstm,42,5000,ok,0.9574,0.7237,0.0575,0.9861,0.1086,654.7028,0.0071
3,smote_cnn_lstm_w1_n5000_seed42,smote,cnn_lstm,42,5000,ok,0.9616,0.7978,0.3564,1.0000,0.5255,628.6074,0.0071
4,adasyn_cnn_lstm_w1_n5000_seed42,adasyn,cnn_lstm,42,5000,ok,0.9697,0.7813,0.1722,1.0000,0.2939,377.0511,0.0071
5,borderline_smote_cnn_lstm_w1_n5000_seed42,borderline_smote,cnn_lstm,42,5000,ok,0.9577,0.7433,0.2024,0.4722,0.2833,252.9794,0.0070
6,main_ctgan_cnn_w1_n5000_seed42_ablation,main_ctgan,cnn,42,5000,ok,0.9618,0.7854,0.2846,1.0000,0.4431,296.8886,0.0069
7,main_ctgan_lstm_w1_n5000_seed42_ablation,main_ctgan,lstm,42,5000,ok,0.9483,0.6943,0.0090,0.4722,0.0178,236.2517,0.0083
8,main_ctgan_cnn_lstm_w0_n5000_seed42_noweight,main_ctgan,cnn_lstm,42,5000,ok,0.9710,0.8829,0.8161,0.9861,0.8931,1225.8388,0.0071
9,main_ctgan_cnn_lstm_w1_n1000_seed42_nablate,main_ctgan,cnn_lstm,42,1000,ok,0.9679,0.7777,0.2028,1.0000,0.3372,547.5226,0.0069


File hasil ada di Drive: /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results/all_runs.csv
Jika sesi akan habis, aman memutus sekarang. Sambungkan lagi dan jalankan sel dari atas; sisa run akan dilanjutkan.


In [27]:
# ============================================================
# (d) Mean ± std antar seed, uji t berpasangan dan Wilcoxon
# Dibandingkan ke main_ctgan (arch cnn_lstm, class weight, n_synth default)
# ============================================================
runs = pd.read_csv(RESULTS_DIR / "all_runs.csv") if (RESULTS_DIR / "all_runs.csv").exists() else pd.DataFrame(all_rows)
if "status" in runs.columns:
    ok = runs[runs["status"].eq("ok")].copy()
else:
    ok = runs.copy()


def summarize_seed_groups(df):
    keys = ["scenario", "arch", "use_class_weight", "n_synth"]
    for k in keys:
        if k not in df.columns:
            df[k] = np.nan
    metrics = [c for c in [
        "accuracy", "f1_macro", "f1_weighted",
        "Web Attack__precision", "Web Attack__recall", "Web Attack__f1",
        "train_seconds", "inference_ms_per_sample",
    ] if c in df.columns]
    rows = []
    for key, g in df.groupby(keys, dropna=False):
        rec = dict(zip(keys, key))
        rec["n_seeds"] = int(g["seed"].nunique()) if "seed" in g.columns else len(g)
        rec["seeds"] = ",".join(map(str, sorted(g["seed"].unique()))) if "seed" in g.columns else ""
        for m in metrics:
            rec[f"{m}_mean"] = float(g[m].mean())
            rec[f"{m}_std"] = float(g[m].std(ddof=1)) if len(g) > 1 else 0.0
        rows.append(rec)
    out = pd.DataFrame(rows)
    out.to_csv(RESULTS_DIR / "seed_summary_mean_std.csv", index=False)
    return out


summary_tbl = summarize_seed_groups(ok) if len(ok) else pd.DataFrame()
print("Ringkasan mean ± std:")
display(summary_tbl)


def paired_tests(df, metric, baseline_scenario="main_ctgan"):
    """Uji berpasangan antar seed: skenario lain vs main_ctgan (cnn_lstm, weight=True, n default)."""
    arch = df["arch"] if "arch" in df.columns else "cnn_lstm"
    nsy = df["n_synth"] if "n_synth" in df.columns else MODE["n_synth"]
    def _is_true(col):
        return col.astype(str).str.lower().isin(["true", "1", "yes"])
    if "use_class_weight" in df.columns:
        wcol = _is_true(df["use_class_weight"])
    else:
        wcol = True
    if "arch" in df.columns:
        arch = df["arch"]
    base_mask = (
        (df["scenario"] == baseline_scenario)
        & (arch == "cnn_lstm")
        & wcol
        & (nsy == MODE["n_synth"])
    )
    # filter tag ablasi: run name tanpa _ablation/_noweight/_nablate lebih aman lewat kolom
    base = df[base_mask].drop_duplicates("seed").set_index("seed")
    rows = []
    for scen, g in df.groupby("scenario"):
        if scen == baseline_scenario:
            continue
        # hanya konfigurasi sebanding (cnn_lstm, weight on, n default) jika kolom ada
        gg = g.copy()
        if "arch" in gg.columns:
            gg = gg[gg["arch"] == "cnn_lstm"]
        if "use_class_weight" in gg.columns:
            gg = gg[gg["use_class_weight"].astype(str).str.lower().isin(["true", "1", "yes"])]
        if "n_synth" in gg.columns:
            gg = gg[gg["n_synth"] == MODE["n_synth"]]
        gg = gg.drop_duplicates("seed").set_index("seed")
        common = sorted(set(base.index) & set(gg.index))
        if len(common) < 2 or metric not in base.columns or metric not in gg.columns:
            rows.append({
                "scenario": scen, "metric": metric, "n_pairs": len(common),
                "status": "skipped_too_few_pairs_or_missing_metric",
            })
            continue
        a = base.loc[common, metric].astype(float).to_numpy()
        b = gg.loc[common, metric].astype(float).to_numpy()
        rec = {
            "scenario": scen, "metric": metric, "n_pairs": len(common),
            "baseline_mean": float(a.mean()), "other_mean": float(b.mean()),
            "mean_diff_other_minus_main": float((b - a).mean()),
        }
        try:
            tstat, tp = scipy_stats.ttest_rel(b, a)
            rec["t_stat"] = float(tstat)
            rec["t_pvalue"] = float(tp)
        except Exception as exc:
            rec["t_error"] = str(exc)
        try:
            wstat, wp = scipy_stats.wilcoxon(b, a, zero_method="wilcox", alternative="two-sided")
            rec["wilcoxon_stat"] = float(wstat)
            rec["wilcoxon_pvalue"] = float(wp)
        except Exception as exc:
            rec["wilcoxon_error"] = str(exc)
            rec["wilcoxon_note"] = "Wilcoxon sering gagal jika n_pairs < 10 atau semua selisih nol."
        rows.append(rec)
    return pd.DataFrame(rows)


test_frames = []
if len(ok) and "seed" in ok.columns:
    for metric in ["f1_macro", "Web Attack__f1"]:
        if metric in ok.columns:
            test_frames.append(paired_tests(ok, metric))
if test_frames:
    tests = pd.concat(test_frames, ignore_index=True)
    tests.to_csv(RESULTS_DIR / "paired_tests_vs_main_ctgan.csv", index=False)
    display(tests)
    print("Catatan: 5 seed adalah sampel kecil; p-value harus ditafsirkan hati-hati. Jangan mengisi tabel ini secara manual.")
else:
    print("Uji berpasangan dilewati (data seed belum cukup).")


Ringkasan mean ± std:


,scenario,arch,use_class_weight,n_synth,n_seeds,seeds,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,f1_weighted_mean,f1_weighted_std,Web Attack__precision_mean,Web Attack__precision_std,Web Attack__recall_mean,Web Attack__recall_std,Web Attack__f1_mean,Web Attack__f1_std,train_seconds_mean,train_seconds_std,inference_ms_per_sample_mean,inference_ms_per_sample_std
0,adasyn,cnn_lstm,True,5000,1,42,0.9697,0.0000,0.7813,0.0000,0.9688,0.0000,0.1722,0.0000,1.0000,0.0000,0.2939,0.0000,377.0511,0.0000,0.0071,0.0000
1,borderline_smote,cnn_lstm,True,5000,1,42,0.9577,0.0000,0.7433,0.0000,0.9541,0.0000,0.2024,0.0000,0.4722,0.0000,0.2833,0.0000,252.9794,0.0000,0.0070,0.0000
2,main_ctgan,cnn,True,5000,1,42,0.9618,0.0000,0.7854,0.0000,0.9599,0.0000,0.2846,0.0000,1.0000,0.0000,0.4431,0.0000,296.8886,0.0000,0.0069,0.0000
3,main_ctgan,cnn_lstm,False,5000,1,42,0.9710,0.0000,0.8829,0.0000,0.9698,0.0000,0.8161,0.0000,0.9861,0.0000,0.8931,0.0000,1225.8388,0.0000,0.0071,0.0000
4,main_ctgan,cnn_lstm,True,1000,1,42,0.9679,0.0000,0.7777,0.0000,0.9658,0.0000,0.2028,0.0000,1.0000,0.0000,0.3372,0.0000,547.5226,0.0000,0.0069,0.0000
5,main_ctgan,cnn_lstm,True,2500,1,42,0.9699,0.0000,0.7976,0.0000,0.9689,0.0000,0.2432,0.0000,1.0000,0.0000,0.3913,0.0000,770.6207,0.0000,0.0071,0.0000
6,main_ctgan,cnn_lstm,True,5000,5,"42,123,456,789,1024",0.9677,0.0033,0.8176,0.0334,0.9666,0.0037,0.3973,0.1900,1.0000,0.0000,0.5465,0.2050,967.0676,271.8491,0.0070,0.0002
7,main_ctgan,cnn_lstm,True,10000,1,42,0.9695,0.0000,0.8176,0.0000,0.9680,0.0000,0.3655,0.0000,1.0000,0.0000,0.5353,0.0000,603.5428,0.0000,0.0071,0.0000
8,main_ctgan,lstm,True,5000,1,42,0.9483,0.0000,0.6943,0.0000,0.9477,0.0000,0.0090,0.0000,0.4722,0.0000,0.0178,0.0000,236.2517,0.0000,0.0083,0.0000
9,smote,cnn_lstm,True,5000,5,"42,123,456,789,1024",0.9678,0.0035,0.7921,0.0082,0.9666,0.0041,0.2450,0.0684,1.0000,0.0000,0.3899,0.0849,591.0523,129.8928,0.0069,0.0002


,scenario,metric,n_pairs,status,baseline_mean,other_mean,mean_diff_other_minus_main,t_stat,t_pvalue,wilcoxon_stat,wilcoxon_pvalue
0,adasyn,f1_macro,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,borderline_smote,f1_macro,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,smote,f1_macro,5,NaN,0.8176,0.7921,-0.0254,-1.4462,0.2217,3.0000,0.3125
3,undersample_only,f1_macro,5,NaN,0.8176,0.7239,-0.0936,-3.2113,0.0325,1.0000,0.1250
4,weights_only,f1_macro,5,NaN,0.8176,0.7288,-0.0888,-4.0223,0.0158,0.0000,0.0625
5,adasyn,Web Attack__f1,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,borderline_smote,Web Attack__f1,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,smote,Web Attack__f1,5,NaN,0.5465,0.3899,-0.1566,-1.6979,0.1648,2.0000,0.1875
8,undersample_only,Web Attack__f1,5,NaN,0.5465,0.1429,-0.4035,-3.0578,0.0377,1.0000,0.1250
9,weights_only,Web Attack__f1,5,NaN,0.5465,0.1195,-0.4270,-3.8992,0.0176,0.0000,0.0625


Catatan: 5 seed adalah sampel kecil; p-value harus ditafsirkan hati-hati. Jangan mengisi tabel ini secara manual.


In [28]:
# Sel ini boleh dijalankan di sesi berikutnya: baca proba_*.npz dari Drive,
# tanpa melatih ulang. Pastikan sel mount, konfigurasi, import, dan praproses/split sudah dijalankan.
# ============================================================
# (e) Ambang keputusan Web Attack, kurva PR, filter tahap kedua
# Dipakai pada proba model utama (seed pertama) jika ada.
# ============================================================
def load_primary_proba():
    # cari file proba main_ctgan cnn_lstm seed pertama, n default, tanpa tag ablasi
    seed = MODE["seeds"][0]
    n = MODE["n_synth"]
    expected = RESULTS_DIR / f"proba_main_ctgan_cnn_lstm_w1_n{n}_seed{seed}.npz"
    if expected.exists():
        data = np.load(expected, allow_pickle=True)
        return data, expected
    cands = sorted(RESULTS_DIR.glob("proba_main_ctgan_cnn_lstm_w1_*.npz"))
    cands = [p for p in cands if "_ablation" not in p.name and "_noweight" not in p.name and "_nablate" not in p.name]
    if not cands:
        print("Tidak ada file proba model utama.")
        return None, None
    print("Memakai", cands[0])
    return np.load(cands[0], allow_pickle=True), cands[0]


def apply_web_threshold(proba, classes, threshold):
    """Prediksi Web Attack hanya jika p_web >= threshold; jika tidak, argmax kelas lain."""
    classes = list(classes)
    if "Web Attack" not in classes:
        return proba.argmax(axis=1)
    w = classes.index("Web Attack")
    p_web = proba[:, w]
    alt = proba.copy()
    alt[:, w] = -np.inf
    pred = np.where(p_web >= threshold, w, alt.argmax(axis=1))
    return pred, p_web


if RUN_THRESHOLD:
    packed, proba_path = load_primary_proba()
    if packed is None:
        print("Analisis ambang dilewati.")
    else:
        proba = packed["proba"]
        y_true = packed["y_true"]
        classes = list(packed["classes"])
        # Kurva PR: Web Attack vs rest
        if "Web Attack" in classes:
            w = classes.index("Web Attack")
            y_bin = (y_true == w).astype(int)
            p_web = proba[:, w]
            prec_c, rec_c, thr_c = precision_recall_curve(y_bin, p_web)
            ap = average_precision_score(y_bin, p_web)
            pr_df = pd.DataFrame({"precision": prec_c, "recall": rec_c,
                                  "threshold": np.append(thr_c, np.nan)})
            pr_df.to_csv(RESULTS_DIR / "pr_curve_web_attack.csv", index=False)
            plt.figure(figsize=(6, 5))
            plt.plot(rec_c, prec_c, label=f"Web Attack AP={ap:.4f}")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.title("Precision–Recall curve — Web Attack (test)")
            plt.legend()
            plt.grid(True, linestyle=":", alpha=0.7)
            plt.tight_layout()
            plt.savefig(RESULTS_DIR / "pr_curve_web_attack.png", dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Average precision Web Attack = {ap:.4f}")

            # Ambang terbaik menurut F1 biner Web vs rest, dan tabel 0.5–0.9
            # Vektorisasi: F1 dihitung langsung dari kurva PR (cepat, tanpa loop)
            if len(thr_c):
                f1_bins = 2 * prec_c[:-1] * rec_c[:-1] / np.clip(prec_c[:-1] + rec_c[:-1], 1e-12, None)
                best_i = int(np.nanargmax(f1_bins))
                best_t = float(thr_c[best_i])
                best_f1 = float(f1_bins[best_i])
            else:
                best_t, best_f1 = 0.5, np.nan
            print(f"Ambang F1-biner terbaik (Web vs rest): {best_t:.4f} (F1={best_f1:.4f})")

            thr_rows = []
            for t in THRESHOLDS + [best_t]:
                pred, _ = apply_web_threshold(proba, classes, t)
                rec, _ = metrics_from_preds(y_true, pred, classes)
                rec["threshold"] = float(t)
                rec["is_best_binary_f1"] = abs(float(t) - best_t) < 1e-12
                rec["average_precision"] = float(ap)
                thr_rows.append(rec)
            thr_df = pd.DataFrame(thr_rows).drop_duplicates(subset=["threshold"])
            cols = ["threshold", "accuracy", "f1_macro",
                    "Web Attack__precision", "Web Attack__recall", "Web Attack__f1",
                    "is_best_binary_f1", "average_precision"]
            thr_df[[c for c in cols if c in thr_df.columns]].to_csv(
                RESULTS_DIR / "threshold_web_attack.csv", index=False
            )
            display(thr_df[[c for c in cols if c in thr_df.columns]])

            plt.figure(figsize=(7, 5))
            plt.plot(thr_df["threshold"], thr_df["Web Attack__precision"], marker="o", label="Precision")
            plt.plot(thr_df["threshold"], thr_df["Web Attack__recall"], marker="o", label="Recall")
            plt.plot(thr_df["threshold"], thr_df["Web Attack__f1"], marker="o", label="F1")
            plt.axvline(0.7, color="gray", linestyle="--", label="0.7 (saran reviewer)")
            plt.xlabel("Decision threshold for Web Attack")
            plt.ylabel("Score")
            plt.legend()
            plt.grid(True, linestyle=":", alpha=0.7)
            plt.tight_layout()
            plt.savefig(RESULTS_DIR / "threshold_web_attack.png", dpi=300, bbox_inches="tight")
            plt.close()
        packed = packed  # keep
else:
    packed = None
    y_true = None
    classes = None
    proba = None


Average precision Web Attack = 0.9515
Ambang F1-biner terbaik (Web vs rest): 0.9984 (F1=0.9524)


,threshold,accuracy,f1_macro,Web Attack__precision,Web Attack__recall,Web Attack__f1,is_best_binary_f1,average_precision
0,0.5000,0.9625,0.8406,0.6207,1.0000,0.7660,False,0.9515
1,0.5500,0.9625,0.8406,0.6207,1.0000,0.7660,False,0.9515
2,0.6000,0.9625,0.8395,0.6174,0.9861,0.7594,False,0.9515
3,0.6500,0.9625,0.8423,0.6396,0.9861,0.7760,False,0.9515
4,0.7000,0.9625,0.8423,0.6396,0.9861,0.7760,False,0.9515
5,0.7500,0.9625,0.8459,0.6698,0.9861,0.7978,False,0.9515
6,0.8000,0.9625,0.8474,0.6827,0.9861,0.8068,False,0.9515
7,0.8500,0.9625,0.8482,0.6893,0.9861,0.8114,False,0.9515
8,0.9000,0.9625,0.8482,0.6893,0.9861,0.8114,False,0.9515
9,0.9984,0.9625,0.8717,0.9333,0.9722,0.9524,True,0.9515


In [29]:
# Membutuhkan X_train_mm dan y_train (dari sel praproses) plus proba model utama di Drive.
# ============================================================
# Filter tahap kedua: RF / LightGBM pada sampel yang diprediksi Web Attack
# Dilatih hanya pada data TRAIN (skala MinMax, label asli, tanpa test).
# ============================================================
if RUN_SECONDARY_FILTER and packed is not None and "Web Attack" in list(packed["classes"]):
    classes = list(packed["classes"])
    w = classes.index("Web Attack")
    proba = packed["proba"]
    y_true = packed["y_true"]
    y_pred_base = proba.argmax(axis=1)

    # Latih filter biner Web vs rest pada train asli (bukan test, bukan sintetis wajib)
    # Pakai train MinMax utuh agar filter melihat Benign asli yang sering jadi FP.
    y_bin_tr = (y_train.values == "Web Attack").astype(int)
    if LGBM_AVAILABLE:
        filter_clf = LGBMClassifier(
            n_estimators=200, learning_rate=0.05, random_state=SPLIT_SEED,
            class_weight="balanced", n_jobs=-1, verbose=-1,
        )
        filter_name = "lightgbm"
    else:
        filter_clf = RandomForestClassifier(
            n_estimators=200, random_state=SPLIT_SEED, class_weight="balanced", n_jobs=-1,
        )
        filter_name = "random_forest"
    t0 = time.perf_counter()
    filter_clf.fit(X_train_mm, y_bin_tr)
    print(f"Filter {filter_name} dilatih dalam {time.perf_counter()-t0:.1f}s pada {len(X_train_mm):,} baris train.")

    pred_filter_test = filter_clf.predict(X_test_mm)
    # Strategi: jika CNN-LSTM prediksi Web Attack tetapi filter bilang bukan, pindahkan ke kelas ke-2 CNN-LSTM
    alt = proba.copy()
    alt[:, w] = -np.inf
    second = alt.argmax(axis=1)
    y_pred_new = y_pred_base.copy()
    mask = (y_pred_base == w) & (pred_filter_test == 0)
    y_pred_new[mask] = second[mask]

    rows = []
    for name, yp in [("cnn_lstm_only", y_pred_base), (f"cnn_lstm_then_{filter_name}", y_pred_new)]:
        rec, _ = metrics_from_preds(y_true, yp, classes)
        rec["pipeline"] = name
        rec["n_web_pred"] = int((yp == w).sum())
        rec["n_rerouted"] = int(mask.sum()) if name != "cnn_lstm_only" else 0
        rows.append(rec)
        save_confusion(y_true, yp, classes, RESULTS_DIR / f"cm_{name}", f"CM — {name}")
    filt_df = pd.DataFrame(rows)
    keep = ["pipeline", "accuracy", "f1_macro",
            "Web Attack__precision", "Web Attack__recall", "Web Attack__f1",
            "n_web_pred", "n_rerouted"]
    filt_df[[c for c in keep if c in filt_df.columns]].to_csv(
        RESULTS_DIR / "secondary_filter_web_attack.csv", index=False
    )
    display(filt_df[[c for c in keep if c in filt_df.columns]])
else:
    print("Filter tahap kedua dilewati.")


Filter lightgbm dilatih dalam 21.7s pada 2,509,032 baris train.


,pipeline,accuracy,f1_macro,Web Attack__precision,Web Attack__recall,Web Attack__f1,n_web_pred,n_rerouted
0,cnn_lstm_only,0.9625,0.8406,0.6207,1.0000,0.7660,116,0
1,cnn_lstm_then_lightgbm,0.9625,0.8773,0.9730,1.0000,0.9863,74,42


In [30]:
# ============================================================
# (f, opsional) Variasi hiperparameter CTGAN — matikan secara default
# ============================================================
if RUN_CTGAN_VARIANT and not IS_SMOKE and CTGAN_AVAILABLE:
    variants = [
        dict(tag="epochs400", epochs=400),
        dict(tag="lr_1e4", learning_rate=1e-4),
        dict(tag="dims128", generator_dim=(128, 128), discriminator_dim=(128, 128)),
    ]
    # di reduced, jalankan satu varian saja
    if RUN_MODE != "full":
        variants = variants[:1]
    for v in variants:
        tag = v.pop("tag")
        try:
            Xs, meta = get_or_fit_ctgan_synth(MODE["n_synth"], MODE["seeds"][0], tag=tag, **v)
            evaluate_ctgan_quality(X_web_gan, Xs, tag=f"variant_{tag}", seed=MODE["seeds"][0])
            # uji cepat: latih model utama sekali (mahal)
            rec, *_ = run_one("main_ctgan", seed=MODE["seeds"][0], tag_extra=f"_ctgan_{tag}")
            # run_one akan fit CTGAN lagi kecuali cache ada — cache sudah ditulis get_or_fit
        except Exception:
            traceback.print_exc()
else:
    print("Variasi hiperparameter CTGAN tidak dijalankan (RUN_CTGAN_VARIANT=False atau mode smoke / tanpa CTGAN).")

# TSTR ringkas: latih RF pada (sintetis Web + non-Web train), uji pada test asli
tstr_path = RESULTS_DIR / "tstr_rf_web.csv"
try:
    cache_main = list(RESULTS_DIR.glob(f"X_web_synth_n{MODE['n_synth']}_seed{MODE['seeds'][0]}_n{MODE['n_synth']}.csv"))
    if not cache_main:
        cache_main = list(RESULTS_DIR.glob("X_web_synth_*.csv"))
    if cache_main:
        Xs = pd.read_csv(cache_main[0])
        X_neg = X_train_mm[y_train.values != "Web Attack"]
        X_tstr = pd.concat([X_neg, Xs[X_neg.columns]], ignore_index=True)
        y_tstr = pd.concat([
            y_train[y_train.values != "Web Attack"],
            pd.Series(["Web Attack"] * len(Xs)),
        ], ignore_index=True)
        rf = RandomForestClassifier(n_estimators=150, random_state=SPLIT_SEED, n_jobs=-1, class_weight="balanced")
        t0 = time.perf_counter()
        rf.fit(X_tstr, y_tstr)
        pred = rf.predict(X_test_mm)
        le_tmp = LabelEncoder()
        le_tmp.fit(CLASS_ORDER if set(CLASS_ORDER) >= set(y_tstr.unique()) | set(y_test.unique()) else sorted(set(y_tstr) | set(y_test)))
        rec, _ = metrics_from_preds(le_tmp.transform(y_test), le_tmp.transform(pred), list(le_tmp.classes_))
        rec["fit_seconds"] = time.perf_counter() - t0
        rec["synth_file"] = str(cache_main[0])
        rec["note"] = "TSTR: RF dilatih pada non-Web train + Web sintetis, diuji pada test asli."
        pd.DataFrame([rec]).to_csv(tstr_path, index=False)
        print("TSTR RF macro F1=", rec.get("f1_macro"), "Web P/R/F1=",
              rec.get("Web Attack__precision"), rec.get("Web Attack__recall"), rec.get("Web Attack__f1"))
    else:
        print("TSTR dilewati: belum ada file sintetis.")
except Exception:
    traceback.print_exc()


Variasi hiperparameter CTGAN tidak dijalankan (RUN_CTGAN_VARIANT=False atau mode smoke / tanpa CTGAN).
TSTR RF macro F1= 0.7373261367691374 Web P/R/F1= 0.0 0.0 0.0


In [31]:
# ============================================================
# Tabel siap naskah (CSV + Markdown + LaTeX) dan gambar 300 dpi
# Termasuk ulang Fig. hasil model utama: CM, riwayat pelatihan,
# P/R/F1 per kelas, dan distribusi kelas sebelum/sesudah CTGAN.
# Tidak mengarang angka: semua merujuk all_runs.csv / proba / cache yang ada.
# ============================================================
from textwrap import dedent

runs_path = RESULTS_DIR / "all_runs.csv"
if not runs_path.exists():
    raise FileNotFoundError(f"Belum ada {runs_path}. Jalankan dulu sel pelatihan, atau sambungkan sesi yang sudah menyimpan hasil ke Drive.")

runs = pd.read_csv(runs_path)
ok = runs[runs["status"].eq("ok")].copy() if "status" in runs.columns else runs.copy()
print(f"all_runs.csv: {len(runs)} baris, ok={len(ok)}, mode tercatat: "
      f"{sorted(ok['run_mode'].astype(str).unique()) if 'run_mode' in ok.columns else '?'}")

# Pastikan ringkasan seed dan uji berpasangan tertulis (sel sebelumnya); buat ulang dari CSV
if "summarize_seed_groups" in dir():
    summary_tbl = summarize_seed_groups(ok) if len(ok) else pd.DataFrame()
else:
    summary_tbl = pd.read_csv(RESULTS_DIR / "seed_summary_mean_std.csv") if (RESULTS_DIR / "seed_summary_mean_std.csv").exists() else pd.DataFrame()

if "paired_tests" in dir() and len(ok) and "seed" in ok.columns:
    test_frames = []
    for metric in ["f1_macro", "Web Attack__f1"]:
        if metric in ok.columns:
            test_frames.append(paired_tests(ok, metric))
    tests = pd.concat(test_frames, ignore_index=True) if test_frames else pd.DataFrame()
    if len(tests):
        tests.to_csv(RESULTS_DIR / "paired_tests_vs_main_ctgan.csv", index=False)
else:
    tests = pd.read_csv(RESULTS_DIR / "paired_tests_vs_main_ctgan.csv") if (RESULTS_DIR / "paired_tests_vs_main_ctgan.csv").exists() else pd.DataFrame()


def _boolish(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _main_mask(df):
    m = (df["scenario"] == "main_ctgan")
    if "arch" in df.columns:
        m &= df["arch"].eq("cnn_lstm")
    if "use_class_weight" in df.columns:
        m &= _boolish(df["use_class_weight"])
    if "n_synth" in df.columns:
        m &= df["n_synth"].eq(MODE["n_synth"])
    if "run_name" in df.columns:
        m &= ~df["run_name"].astype(str).str.contains("_ablation|_noweight|_nablate|_ctgan_", regex=True)
    return m


def fmt_mean_std(mean, std, nd=4):
    if pd.isna(mean):
        return "—"
    if pd.isna(std) or std == 0:
        return f"{mean:.{nd}f}"
    return f"{mean:.{nd}f} ± {std:.{nd}f}"


def to_markdown(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |",
             "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in row.tolist()) + " |")
    return "\n".join(lines)


def to_latex(df, caption, label):
    cols = list(df.columns)
    colspec = "l" + "c" * (len(cols) - 1)
    body = []
    body.append("\\begin{table}[ht]")
    body.append("\\centering")
    body.append(f"\\caption{{{caption}}}")
    body.append(f"\\label{{{label}}}")
    body.append("\\begin{tabular}{" + colspec + "}")
    body.append("\\hline")
    body.append(" & ".join(map(str, cols)) + " \\\\")
    body.append("\\hline")
    for _, row in df.iterrows():
        vals = []
        for v in row.tolist():
            s = "" if pd.isna(v) else str(v)
            s = s.replace("_", "\\_").replace("%", "\\%")
            vals.append(s)
        body.append(" & ".join(vals) + " \\\\")
    body.append("\\hline")
    body.append("\\end{tabular}")
    body.append("\\end{table}")
    return "\n".join(body)


# ----- Table 14 style: baselines on seed 0, plus mean±std if 5 seeds exist -----
seed0 = MODE["seeds"][0]
base_scens = ["weights_only", "undersample_only", "smote", "adasyn", "borderline_smote", "main_ctgan"]
pretty = {
    "weights_only": "No augmentation (class weights only)",
    "undersample_only": "Undersample only",
    "smote": "SMOTE",
    "adasyn": "ADASYN",
    "borderline_smote": "Borderline-SMOTE",
    "main_ctgan": "Targeted CTGAN (this study)",
}

def _comparable(df, scen):
    g = df[df["scenario"] == scen].copy()
    if "arch" in g.columns:
        g = g[g["arch"] == "cnn_lstm"]
    if "use_class_weight" in g.columns:
        g = g[_boolish(g["use_class_weight"])]
    if "n_synth" in g.columns:
        g = g[g["n_synth"] == MODE["n_synth"]]
    if "run_name" in g.columns:
        g = g[~g["run_name"].astype(str).str.contains("_ablation|_noweight|_nablate|_ctgan_", regex=True)]
    return g


rows14 = []
for scen in base_scens:
    g = _comparable(ok, scen)
    if not len(g):
        continue
    g0 = g[g["seed"] == seed0] if "seed" in g.columns else g
    rec = {"Method": pretty.get(scen, scen), "n_seeds": int(g["seed"].nunique()) if "seed" in g.columns else len(g)}
    for src, dst in [("accuracy", "Accuracy"), ("f1_macro", "Macro F-score"),
                     ("Web Attack__precision", "WA precision"),
                     ("Web Attack__recall", "WA recall"),
                     ("Web Attack__f1", "WA F-score")]:
        if src not in g.columns:
            rec[dst] = "—"
            rec[dst + " (seed 42)"] = "—"
            continue
        rec[dst] = fmt_mean_std(g[src].mean(), g[src].std(ddof=1) if len(g) > 1 else 0.0)
        if len(g0):
            rec[dst + " (seed 42)"] = f"{float(g0.iloc[0][src]):.4f}"
        else:
            rec[dst + " (seed 42)"] = "—"
    rows14.append(rec)
table14 = pd.DataFrame(rows14)
table14.to_csv(RESULTS_DIR / "paper_table_baselines.csv", index=False)

# ----- Table 15 style: ablations (seed 0) -----
ablation_specs = [
    ("main_ctgan", "cnn_lstm", True, MODE["n_synth"], "", "Full model (CTGAN + CNN-LSTM + weights)"),
    ("undersample_only", "cnn_lstm", True, MODE["n_synth"], "", "Without CTGAN (undersample only)"),
    ("main_ctgan", "cnn", True, MODE["n_synth"], "_ablation", "CNN only (no LSTM)"),
    ("main_ctgan", "lstm", True, MODE["n_synth"], "_ablation", "LSTM only (no CNN)"),
    ("main_ctgan", "cnn_lstm", False, MODE["n_synth"], "_noweight", "Without class weights"),
]
rows15 = []
for scen, arch, w, n, tag, label in ablation_specs:
    name = make_run_name(scen, seed0, arch, w, n, tag)
    g = ok[ok["run_name"] == name] if "run_name" in ok.columns else pd.DataFrame()
    rec = {"Variant": label, "run_name": name}
    if len(g):
        r = g.iloc[0]
        rec["Accuracy"] = f"{float(r.get('accuracy', float('nan'))):.4f}"
        rec["Macro F-score"] = f"{float(r.get('f1_macro', float('nan'))):.4f}"
        rec["WA F-score"] = f"{float(r.get('Web Attack__f1', float('nan'))):.4f}"
        rec["WA precision"] = f"{float(r.get('Web Attack__precision', float('nan'))):.4f}"
        rec["WA recall"] = f"{float(r.get('Web Attack__recall', float('nan'))):.4f}"
    else:
        rec.update({k: "—" for k in ["Accuracy", "Macro F-score", "WA F-score", "WA precision", "WA recall"]})
    rows15.append(rec)
# synth-count variants
if RUN_ABLATION_SYNTH_N or True:
    for n in MODE["synth_counts"]:
        tag = "" if n == MODE["n_synth"] else "_nablate"
        name = make_run_name("main_ctgan", seed0, "cnn_lstm", True, n, tag)
        g = ok[ok["run_name"] == name] if "run_name" in ok.columns else pd.DataFrame()
        rec = {"Variant": f"Synthetic count N={n}", "run_name": name}
        if len(g):
            r = g.iloc[0]
            rec["Accuracy"] = f"{float(r.get('accuracy', float('nan'))):.4f}"
            rec["Macro F-score"] = f"{float(r.get('f1_macro', float('nan'))):.4f}"
            rec["WA F-score"] = f"{float(r.get('Web Attack__f1', float('nan'))):.4f}"
            rec["WA precision"] = f"{float(r.get('Web Attack__precision', float('nan'))):.4f}"
            rec["WA recall"] = f"{float(r.get('Web Attack__recall', float('nan'))):.4f}"
        else:
            rec.update({k: "—" for k in ["Accuracy", "Macro F-score", "WA F-score", "WA precision", "WA recall"]})
        # avoid duplicating the full model row when n == default (already first row)
        if n != MODE["n_synth"]:
            rows15.append(rec)
table15 = pd.DataFrame(rows15)
table15.to_csv(RESULTS_DIR / "paper_table_ablation.csv", index=False)

# ----- Table 16 style: five-seed mean ± std + tests -----
rows16 = []
for scen in ["main_ctgan", "weights_only", "undersample_only", "smote"]:
    g = _comparable(ok, scen)
    if not len(g):
        continue
    rec = {"Model": pretty.get(scen, scen), "n_seeds": int(g["seed"].nunique()) if "seed" in g.columns else len(g)}
    rec["Macro F-score (mean ± std)"] = fmt_mean_std(
        g["f1_macro"].mean() if "f1_macro" in g.columns else np.nan,
        g["f1_macro"].std(ddof=1) if "f1_macro" in g.columns and len(g) > 1 else 0.0,
    )
    rec["WA F-score (mean ± std)"] = fmt_mean_std(
        g["Web Attack__f1"].mean() if "Web Attack__f1" in g.columns else np.nan,
        g["Web Attack__f1"].std(ddof=1) if "Web Attack__f1" in g.columns and len(g) > 1 else 0.0,
    )
    rec["seeds"] = ",".join(map(str, sorted(g["seed"].unique()))) if "seed" in g.columns else ""
    rows16.append(rec)
table16 = pd.DataFrame(rows16)
if len(tests):
    tests.to_csv(RESULTS_DIR / "paper_table_paired_tests.csv", index=False)
table16.to_csv(RESULTS_DIR / "paper_table_multiseed.csv", index=False)

print("\n===== Table 14 — baselines (mean ± std jika ada beberapa seed; plus nilai seed 42) =====")
display(table14)
print(to_markdown(table14))
print(to_latex(table14, "Resampling baselines versus targeted CTGAN on the isolated test partition.", "tab:baselines"))

print("\n===== Table 15 — ablation (seed 42) =====")
display(table15)
print(to_markdown(table15))
print(to_latex(table15, "Ablation results on the isolated test partition.", "tab:ablation"))

print("\n===== Table 16 — multi-seed =====")
display(table16)
print(to_markdown(table16))
print(to_latex(table16, "Five-seed summary on the isolated test partition.", "tab:multiseed"))
if len(tests):
    print("\n===== Paired tests versus main CTGAN =====")
    display(tests)
    print(to_markdown(tests))
    print(to_latex(tests, "Paired t-test and Wilcoxon signed-rank tests versus the main CTGAN model.", "tab:paired"))

# ----- Gambar 300 dpi dari model utama -----
FIG_PAPER_DIR.mkdir(parents=True, exist_ok=True)
main_name = make_run_name("main_ctgan", seed0, "cnn_lstm", True, MODE["n_synth"], "")
print("\nModel utama untuk gambar paper:", main_name)

# Riwayat pelatihan (Fig. 9 style)
hist_csv = RESULTS_DIR / f"history_{main_name}.csv"
if hist_csv.exists():
    hdf = pd.read_csv(hist_csv)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    if "accuracy" in hdf.columns:
        axes[0].plot(hdf["accuracy"], label="Training accuracy")
        if "val_accuracy" in hdf.columns:
            axes[0].plot(hdf["val_accuracy"], label="Validation accuracy", linestyle="--")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].legend()
        axes[0].grid(True, linestyle=":", alpha=0.7)
    if "loss" in hdf.columns:
        axes[1].plot(hdf["loss"], label="Training loss")
        if "val_loss" in hdf.columns:
            axes[1].plot(hdf["val_loss"], label="Validation loss", linestyle="--")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].legend()
        axes[1].grid(True, linestyle=":", alpha=0.7)
    fig.suptitle("Training and validation accuracy and loss of the hybrid CNN-LSTM model")
    fig.tight_layout()
    out = FIG_PAPER_DIR / "fig_training_history_300dpi.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    fig.savefig(RESULTS_DIR / "fig_training_history_300dpi.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Tersimpan", out)
else:
    print("Riwayat pelatihan tidak ditemukan:", hist_csv)

# Confusion matrix (Fig. 10 style)
cm_csv = RESULTS_DIR / f"cm_{main_name}.csv"
if cm_csv.exists():
    cmdf = pd.read_csv(cm_csv, index_col=0)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cmdf, annot=True, fmt="d", cmap="Blues")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion matrix on the isolated six-class test partition")
    plt.tight_layout()
    out = FIG_PAPER_DIR / "fig_confusion_matrix_300dpi.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.savefig(RESULTS_DIR / "fig_confusion_matrix_300dpi.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("Tersimpan", out)
else:
    print("CSV confusion matrix tidak ditemukan:", cm_csv)

# Per-class P/R/F1 (Fig. 11 style)
main_rows = ok[ok["run_name"] == main_name] if "run_name" in ok.columns else pd.DataFrame()
if len(main_rows):
    r = main_rows.iloc[0]
    classes = list(CLASS_ORDER)
    prec = [r.get(f"{c}__precision", np.nan) for c in classes]
    reca = [r.get(f"{c}__recall", np.nan) for c in classes]
    f1s = [r.get(f"{c}__f1", np.nan) for c in classes]
    x = np.arange(len(classes))
    w = 0.25
    plt.figure(figsize=(10, 5))
    plt.bar(x - w, prec, width=w, label="Precision")
    plt.bar(x, reca, width=w, label="Recall")
    plt.bar(x + w, f1s, width=w, label="F-score")
    plt.xticks(x, classes, rotation=25, ha="right")
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.title("Per-class precision, recall, and F-score on the isolated test partition")
    plt.legend()
    plt.grid(True, axis="y", linestyle=":", alpha=0.7)
    plt.tight_layout()
    out = FIG_PAPER_DIR / "fig_per_class_prf_300dpi.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.savefig(RESULTS_DIR / "fig_per_class_prf_300dpi.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("Tersimpan", out)
else:
    print("Baris model utama belum ada di all_runs.csv; grafik P/R/F1 dilewati.")

# Distribusi kelas sebelum/sesudah CTGAN (Fig. 5 / Fig. 6 style)
try:
    # sebelum: y_train (split, belum CTGAN); sesudah: bangun ulang dari cache
    before = pd.Series(y_train).value_counts().reindex(CLASS_ORDER)
    Xs, meta = get_or_fit_ctgan_synth(MODE["n_synth"], seed0, tag=f"n{MODE['n_synth']}")
    Xb, yb, _ = build_train_for_scenario("main_ctgan", MODE["n_synth"], seed0, X_synth_ctgan=Xs)
    after = pd.Series(yb).value_counts().reindex(CLASS_ORDER)
    dist = pd.DataFrame({"before_ctgan_train_split": before, "after_ctgan_and_undersample": after})
    dist.to_csv(RESULTS_DIR / "class_counts_before_after_ctgan.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
    # log counts of cleaned six-class table if available
    if "y" in globals() and len(y):
        full = pd.Series(y).value_counts().reindex(CLASS_ORDER)
        axes[0].bar(np.arange(len(CLASS_ORDER)), full.values, color="#4C78A8")
        axes[0].set_yscale("log")
        axes[0].set_xticks(np.arange(len(CLASS_ORDER)))
        axes[0].set_xticklabels(CLASS_ORDER, rotation=25, ha="right")
        axes[0].set_title("Six-class counts in the cleaned table (log scale)")
        axes[0].set_ylabel("Count")
        axes[0].grid(True, axis="y", linestyle=":", alpha=0.7)
        full.to_csv(RESULTS_DIR / "class_counts_cleaned_table.csv")
    else:
        axes[0].set_visible(False)

    x = np.arange(len(CLASS_ORDER))
    w = 0.35
    axes[1].bar(x - w / 2, before.values, width=w, label="Train before CTGAN")
    axes[1].bar(x + w / 2, after.values, width=w, label="Train after CTGAN + undersample")
    axes[1].set_yscale("log")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(CLASS_ORDER, rotation=25, ha="right")
    axes[1].set_title("Training class counts before and after CTGAN (log scale)")
    axes[1].legend()
    axes[1].grid(True, axis="y", linestyle=":", alpha=0.7)
    fig.tight_layout()
    out = FIG_PAPER_DIR / "fig_class_distribution_300dpi.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    fig.savefig(RESULTS_DIR / "fig_class_distribution_300dpi.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Tersimpan", out)
except Exception:
    traceback.print_exc()
    print("Distribusi kelas tidak dapat digambar (cache sintetis atau y_train belum ada).")

# Salin juga kurva PR 300 dpi jika sudah ada dari sel ambang
pr_png = RESULTS_DIR / "pr_curve_web_attack.png"
if pr_png.exists():
    # file itu sudah 300 dpi; salin ke folder paper
    try:
        import shutil
        shutil.copy2(pr_png, FIG_PAPER_DIR / "fig_pr_curve_web_attack_300dpi.png")
        print("Menyalin kurva PR ke figures_paper/")
    except Exception as exc:
        print("Salin kurva PR gagal:", exc)

print("\nSemua tabel/gambar paper ada di:")
print(" ", RESULTS_DIR)
print(" ", FIG_PAPER_DIR)
print("Jangan memakai angka mode smoke. Periksa status=ok dan jumlah seed sebelum menyalin ke naskah.")


all_runs.csv: 28 baris, ok=28, mode tercatat: ['full']

===== Table 14 — baselines (mean ± std jika ada beberapa seed; plus nilai seed 42) =====


,Method,n_seeds,Accuracy,Accuracy (seed 42),Macro F-score,Macro F-score (seed 42),WA precision,WA precision (seed 42),WA recall,WA recall (seed 42),WA F-score,WA F-score (seed 42)
0,No augmentation (class weights only),5,0.9295 ± 0.0284,0.9537,0.7288 ± 0.0476,0.7118,0.1282 ± 0.2680,0.0275,0.8944 ± 0.2360,1.0000,0.1195 ± 0.2312,0.0536
1,Undersample only,5,0.9321 ± 0.0399,0.9574,0.7239 ± 0.0422,0.7237,0.0886 ± 0.0971,0.0575,0.8917 ± 0.2346,0.9861,0.1429 ± 0.1527,0.1086
2,SMOTE,5,0.9678 ± 0.0035,0.9616,0.7921 ± 0.0082,0.7978,0.2450 ± 0.0684,0.3564,1.0000,1.0000,0.3899 ± 0.0849,0.5255
3,ADASYN,1,0.9697,0.9697,0.7813,0.7813,0.1722,0.1722,1.0000,1.0000,0.2939,0.2939
4,Borderline-SMOTE,1,0.9577,0.9577,0.7433,0.7433,0.2024,0.2024,0.4722,0.4722,0.2833,0.2833


| Method | n_seeds | Accuracy | Accuracy (seed 42) | Macro F-score | Macro F-score (seed 42) | WA precision | WA precision (seed 42) | WA recall | WA recall (seed 42) | WA F-score | WA F-score (seed 42) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| No augmentation (class weights only) | 5 | 0.9295 ± 0.0284 | 0.9537 | 0.7288 ± 0.0476 | 0.7118 | 0.1282 ± 0.2680 | 0.0275 | 0.8944 ± 0.2360 | 1.0000 | 0.1195 ± 0.2312 | 0.0536 |
| Undersample only | 5 | 0.9321 ± 0.0399 | 0.9574 | 0.7239 ± 0.0422 | 0.7237 | 0.0886 ± 0.0971 | 0.0575 | 0.8917 ± 0.2346 | 0.9861 | 0.1429 ± 0.1527 | 0.1086 |
| SMOTE | 5 | 0.9678 ± 0.0035 | 0.9616 | 0.7921 ± 0.0082 | 0.7978 | 0.2450 ± 0.0684 | 0.3564 | 1.0000 | 1.0000 | 0.3899 ± 0.0849 | 0.5255 |
| ADASYN | 1 | 0.9697 | 0.9697 | 0.7813 | 0.7813 | 0.1722 | 0.1722 | 1.0000 | 1.0000 | 0.2939 | 0.2939 |
| Borderline-SMOTE | 1 | 0.9577 | 0.9577 | 0.7433 | 0.7433 | 0.2024 | 0.2024 | 0.4722 | 0.4722 | 0.2833 | 0.2833 |
\begin{table}[ht]
\ce

,Variant,run_name,Accuracy,Macro F-score,WA F-score,WA precision,WA recall
0,Full model (CTGAN + CNN-LSTM + weights),main_ctgan_cnn_lstm_w1_n5000_seed42,0.9625,0.8406,0.7660,0.6207,1.0000
1,Without CTGAN (undersample only),undersample_only_cnn_lstm_w1_n5000_seed42,0.9574,0.7237,0.1086,0.0575,0.9861
2,CNN only (no LSTM),main_ctgan_cnn_w1_n5000_seed42_ablation,0.9618,0.7854,0.4431,0.2846,1.0000
3,LSTM only (no CNN),main_ctgan_lstm_w1_n5000_seed42_ablation,0.9483,0.6943,0.0178,0.0090,0.4722
4,Without class weights,main_ctgan_cnn_lstm_w0_n5000_seed42_noweight,0.9710,0.8829,0.8931,0.8161,0.9861
5,Synthetic count N=1000,main_ctgan_cnn_lstm_w1_n1000_seed42_nablate,0.9679,0.7777,0.3372,0.2028,1.0000
6,Synthetic count N=2500,main_ctgan_cnn_lstm_w1_n2500_seed42_nablate,0.9699,0.7976,0.3913,0.2432,1.0000
7,Synthetic count N=10000,main_ctgan_cnn_lstm_w1_n10000_seed42_nablate,0.9695,0.8176,0.5353,0.3655,1.0000


| Variant | run_name | Accuracy | Macro F-score | WA F-score | WA precision | WA recall |
| --- | --- | --- | --- | --- | --- | --- |
| Full model (CTGAN + CNN-LSTM + weights) | main_ctgan_cnn_lstm_w1_n5000_seed42 | 0.9625 | 0.8406 | 0.7660 | 0.6207 | 1.0000 |
| Without CTGAN (undersample only) | undersample_only_cnn_lstm_w1_n5000_seed42 | 0.9574 | 0.7237 | 0.1086 | 0.0575 | 0.9861 |
| CNN only (no LSTM) | main_ctgan_cnn_w1_n5000_seed42_ablation | 0.9618 | 0.7854 | 0.4431 | 0.2846 | 1.0000 |
| LSTM only (no CNN) | main_ctgan_lstm_w1_n5000_seed42_ablation | 0.9483 | 0.6943 | 0.0178 | 0.0090 | 0.4722 |
| Without class weights | main_ctgan_cnn_lstm_w0_n5000_seed42_noweight | 0.9710 | 0.8829 | 0.8931 | 0.8161 | 0.9861 |
| Synthetic count N=1000 | main_ctgan_cnn_lstm_w1_n1000_seed42_nablate | 0.9679 | 0.7777 | 0.3372 | 0.2028 | 1.0000 |
| Synthetic count N=2500 | main_ctgan_cnn_lstm_w1_n2500_seed42_nablate | 0.9699 | 0.7976 | 0.3913 | 0.2432 | 1.0000 |
| Synthetic count N=10000 | main_ctgan

,Model,n_seeds,Macro F-score (mean ± std),WA F-score (mean ± std),seeds
0,No augmentation (class weights only),5,0.7288 ± 0.0476,0.1195 ± 0.2312,"42,123,456,789,1024"
1,Undersample only,5,0.7239 ± 0.0422,0.1429 ± 0.1527,"42,123,456,789,1024"
2,SMOTE,5,0.7921 ± 0.0082,0.3899 ± 0.0849,"42,123,456,789,1024"


| Model | n_seeds | Macro F-score (mean ± std) | WA F-score (mean ± std) | seeds |
| --- | --- | --- | --- | --- |
| No augmentation (class weights only) | 5 | 0.7288 ± 0.0476 | 0.1195 ± 0.2312 | 42,123,456,789,1024 |
| Undersample only | 5 | 0.7239 ± 0.0422 | 0.1429 ± 0.1527 | 42,123,456,789,1024 |
| SMOTE | 5 | 0.7921 ± 0.0082 | 0.3899 ± 0.0849 | 42,123,456,789,1024 |
\begin{table}[ht]
\centering
\caption{Five-seed summary on the isolated test partition.}
\label{tab:multiseed}
\begin{tabular}{lcccc}
\hline
Model & n_seeds & Macro F-score (mean ± std) & WA F-score (mean ± std) & seeds \\
\hline
No augmentation (class weights only) & 5 & 0.7288 ± 0.0476 & 0.1195 ± 0.2312 & 42,123,456,789,1024 \\
Undersample only & 5 & 0.7239 ± 0.0422 & 0.1429 ± 0.1527 & 42,123,456,789,1024 \\
SMOTE & 5 & 0.7921 ± 0.0082 & 0.3899 ± 0.0849 & 42,123,456,789,1024 \\
\hline
\end{tabular}
\end{table}

===== Paired tests versus main CTGAN =====


,scenario,metric,n_pairs,status,baseline_mean,other_mean,mean_diff_other_minus_main,t_stat,t_pvalue,wilcoxon_stat,wilcoxon_pvalue
0,adasyn,f1_macro,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,borderline_smote,f1_macro,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,smote,f1_macro,5,NaN,0.8176,0.7921,-0.0254,-1.4462,0.2217,3.0000,0.3125
3,undersample_only,f1_macro,5,NaN,0.8176,0.7239,-0.0936,-3.2113,0.0325,1.0000,0.1250
4,weights_only,f1_macro,5,NaN,0.8176,0.7288,-0.0888,-4.0223,0.0158,0.0000,0.0625
5,adasyn,Web Attack__f1,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,borderline_smote,Web Attack__f1,1,skipped_too_few_pairs_or_missing_metric,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,smote,Web Attack__f1,5,NaN,0.5465,0.3899,-0.1566,-1.6979,0.1648,2.0000,0.1875
8,undersample_only,Web Attack__f1,5,NaN,0.5465,0.1429,-0.4035,-3.0578,0.0377,1.0000,0.1250
9,weights_only,Web Attack__f1,5,NaN,0.5465,0.1195,-0.4270,-3.8992,0.0176,0.0000,0.0625


| scenario | metric | n_pairs | status | baseline_mean | other_mean | mean_diff_other_minus_main | t_stat | t_pvalue | wilcoxon_stat | wilcoxon_pvalue |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| adasyn | f1_macro | 1 | skipped_too_few_pairs_or_missing_metric |  |  |  |  |  |  |  |
| borderline_smote | f1_macro | 1 | skipped_too_few_pairs_or_missing_metric |  |  |  |  |  |  |  |
| smote | f1_macro | 5 |  | 0.8175650610648605 | 0.792128328810931 | -0.025436732253929395 | -1.4462184714078872 | 0.2216516306044557 | 3.0 | 0.3125 |
| undersample_only | f1_macro | 5 |  | 0.8175650610648605 | 0.7239249252829567 | -0.09364013578190371 | -3.211331529884736 | 0.03254765634411682 | 1.0 | 0.125 |
| weights_only | f1_macro | 5 |  | 0.8175650610648605 | 0.7287535812440243 | -0.08881147982083604 | -4.022344164247064 | 0.015833632145523267 | 0.0 | 0.0625 |
| adasyn | Web Attack__f1 | 1 | skipped_too_few_pairs_or_missing_metric |  |  |  |  |  |  |  |
| borderline_smote | Web 

In [32]:
# ============================================================
# Ringkasan berkas hasil + peringatan mode
# ============================================================
manifest = {
    "run_mode": RUN_MODE,
    "is_smoke_or_no_real_data": bool(IS_SMOKE or RUN_MODE == "smoke"),
    "split_seed_test": SPLIT_SEED,
    "test_untouched": True,
    "finished_iso": datetime.now().isoformat(timespec="seconds"),
    "results_dir": str(RESULTS_DIR.resolve()),
    "files": sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_file()),
}
(RESULTS_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
if manifest["is_smoke_or_no_real_data"]:
    print("\n*** PERINGATAN: mode smoke / data palsu. Jangan masukkan angka ini ke makalah. ***")
else:
    print("\nSelesai. Salin tabel/gambar dari results/ ke naskah hanya setelah memeriksa all_runs.csv.")


{
  "run_mode": "full",
  "is_smoke_or_no_real_data": false,
  "split_seed_test": 42,
  "test_untouched": true,
  "finished_iso": "2026-11-14T10:23:50",
  "results_dir": "/content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results",
  "files": [
    "X_web_gan.csv",
    "X_web_synth_n10000_nablate_seed42_n10000.csv",
    "X_web_synth_n1000_nablate_seed42_n1000.csv",
    "X_web_synth_n2500_nablate_seed42_n2500.csv",
    "X_web_synth_n5000_ablation_seed42_n5000.csv",
    "X_web_synth_n5000_noweight_seed42_n5000.csv",
    "X_web_synth_n5000_seed1024_n5000.csv",
    "X_web_synth_n5000_seed123_n5000.csv",
    "X_web_synth_n5000_seed42_n5000.csv",
    "X_web_synth_n5000_seed456_n5000.csv",
    "X_web_synth_n5000_seed789_n5000.csv",
    "all_runs.csv",
    "class_counts_before_after_ctgan.csv",
    "class_counts_cleaned_table.csv",
    "cm_adasyn_cnn_lstm_w1_n5000_seed42.csv",
    "cm_adasyn_cnn_lstm_w1_n5000_seed42.png",
    "cm_borderline_smote_cnn_lstm_w1_n5000_seed42.csv",
    "cm_borderli

## Catatan untuk penulis

- Test set selalu dari split 80/20 `random_state=42` dan tidak di-augmentasi.
- Seed 123/456/789/1024 mengulang CTGAN/SMOTE, undersample Benign, dan inisialisasi model; **bukan** mengulang pecahan test (agar perbandingan antar seed tetap pada test yang sama, sesuai “untouched test set”). Jika reviewer meminta split berbeda per seed, ubah `split_and_scale(..., split_seed=seed)` secara sadar — itu protokol lain.
- Mode `reduced` memotong epoch/seed; sebutkan ini di respons reviewer jika dipakai, lalu konfirmasi konfigurasi kunci di `full` bila memungkinkan.
- Tidak ada hasil fiktif di notebook ini. Sel yang gagal ditulis ke `all_runs.csv` dengan `status=error`.


## Catatan tambahan untuk Colab

- Setelah disconnect, jalankan lagi semua sel dari *mount Drive*. Pelatihan dengan `status=ok` dilewati; cache CTGAN dan `proba_*.npz` tetap dipakai.
- Analisis ambang, filter tahap kedua, mutu CTGAN, dan tabel paper tidak membutuhkan pelatihan ulang selama file di `results/` utuh.
- Jangan salin `all_runs.csv` dari laptop ke folder Colab jika Anda ingin seluruh angka makalah berasal dari satu mesin.
